# Part 2


## Imports


In [1]:
import os
import re
import sys
import json
import time
import uuid
import zipfile
import sqlite3
import hashlib
import threading
import subprocess
from pathlib import Path
from datetime import date, datetime, timedelta
from collections import Counter
from urllib.parse import urlparse
from xml.etree import ElementTree as ET
from typing import Any, Literal, TypedDict

import duckdb
from firmable.database import connect_database, close_database
from firmable.readers import prepare_csv_input, read_csv_python
import pandas as pd
import numpy as np
import requests
import yaml

from dotenv import load_dotenv
from IPython.display import display
from openai import OpenAI
from pydantic import BaseModel, Field, ConfigDict, ValidationError, model_validator
try:
    from tavily import TavilyClient
except ImportError:
    TavilyClient = None

from langchain.chat_models import init_chat_model
from langchain_core.tools import tool
from langchain_core.messages import AIMessage, ToolMessage, SystemMessage, HumanMessage
from deepagents import create_deep_agent

from langgraph.graph import StateGraph, START, END
from langgraph.types import Command, interrupt

try:
    from langgraph.checkpoint.sqlite import SqliteSaver
except ImportError:
    SqliteSaver = None

try:
    from langgraph.checkpoint.memory import InMemorySaver
except ImportError:
    from langgraph.checkpoint.memory import MemorySaver as InMemorySaver

try:
    from langsmith import Client as LangSmithClient
except ImportError:
    LangSmithClient = None

## Paths


In [2]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
load_dotenv(PROJECT_ROOT / ".env")

PART1_DIR = PROJECT_ROOT / "outputs" / "part1"
PART2_DIR = PROJECT_ROOT / "outputs" / "part2"

RAW_DIR = PART2_DIR / "raw"
PROFILE_DIR = PART2_DIR / "dataset_profiles"
REVIEW_DIR = PART2_DIR / "review"
CONFIG_DIR = PART2_DIR / "mapping_configs"
REJECTED_DIR = PART2_DIR / "rejected"
STATE_DIR = PART2_DIR / "state"
METRICS_DIR = PART2_DIR / "metrics"
SANDBOX_DIR = PART2_DIR / "sandbox"

for path in [
    RAW_DIR,
    PROFILE_DIR,
    REVIEW_DIR,
    CONFIG_DIR,
    REJECTED_DIR,
    STATE_DIR,
    METRICS_DIR,
    SANDBOX_DIR,
]:
    path.mkdir(parents=True, exist_ok=True)

ONTOLOGY_PATH = PROJECT_ROOT / "resources" / "firmable_ontology.yaml"
if not ONTOLOGY_PATH.exists():
    ONTOLOGY_PATH = PROJECT_ROOT / "resouces" / "firmable_ontology.yaml"

DB_PATH = PROJECT_ROOT / "firmable.duckdb"
GOLD_COPY_DIR = PART2_DIR / "gold_inspection"
GOLD_COPY_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_PATH = PART2_DIR / "workflow_checkpoints.sqlite"

CKAN_API = "https://data.gov.au/data/api/3/action"

EXTRACTOR_VERSION = "firmable-part2-v1"
HOLDOUT_SEED = 20260929
VALIDATION_ROWS = 1000
EXTRACT_ROWS = 1000
MAX_NORMAL_REVISIONS = 2
MAX_READER_REVISIONS = 1

NORMAL_MODEL = os.getenv("FIRMABLE_NORMAL_MODEL", "openai:gpt-6-luna")
TURBO_MODEL = os.getenv("FIRMABLE_TURBO_MODEL", "openai:gpt-6.1-sol")

assert ONTOLOGY_PATH.exists(), f"Ontology not found: {ONTOLOGY_PATH}"
assert (PART1_DIR / "ranked_shortlist_50.csv").exists(), "Part 1 shortlist is missing"

con = None
duckdb_lock = threading.Lock()


def get_db():
    global con
    if con is not None:
        try:
            con.execute("SELECT 1")
            return con
        except duckdb.Error:
            close_database(con)
            con = None
    con = connect_database(DB_PATH)
    if con is not None and CTX.get("source_table"):
        try:
            # Closing a failed run removes TEMP tables. Rebuild them for resume.
            if not con.execute("SELECT count(*) FROM information_schema.tables WHERE table_name = 'source_data'").fetchone()[0]:
                _prepare_source_views(CTX["source_table"])
        except duckdb.Error as error:
            print("PART 2 — Source views could not be restored:", error)
            close_db()
    return con


def close_db():
    global con
    close_database(con)
    con = None


duckdb_lock = threading.Lock()

print("Project:", PROJECT_ROOT)
print("Database:", DB_PATH)
print("Normal model:", NORMAL_MODEL)
print("Turbo model:", TURBO_MODEL)

Project: /Users/ario/Documents/firmable
Database: /Users/ario/Documents/firmable/firmable.duckdb
Normal model: openai:gpt-6-luna
Turbo model: openai:gpt-6.1-sol


## API checks


In [3]:
def setup_openai():
    api_key = os.getenv("OPENAI_API_KEY")

    if not api_key:
        print("OpenAI disabled: OPENAI_API_KEY is missing.")
        return None

    client = OpenAI(api_key=api_key)

    try:
        # Authentication probe only. Never compare configured model names
        # against this list because aliases/access may not be listed literally.
        client.models.list()
        return client

    except Exception as e:
        status = getattr(e, "status_code", None)

        if status in {401, 403}:
            print(
                "OpenAI disabled:",
                f"{type(e).__name__}: {e}",
            )
            return None

        # Some proxies/restricted keys do not expose /models even though
        # inference works. Defer the real model check until the first call.
        print(
            "OpenAI key present; model-list probe unavailable. "
            "Inference validation deferred.",
            f"({type(e).__name__})",
        )
        return client


def setup_langsmith():
    os.environ["LANGSMITH_TRACING"] = "false"

    key = os.getenv("LANGSMITH_API_KEY")

    if not key or LangSmithClient is None:
        return None

    try:
        client = LangSmithClient(api_key=key)

        # Small authenticated read. Invalid keys disable tracing only.
        next(iter(client.list_projects(limit=1)), None)

        os.environ["LANGSMITH_TRACING"] = "true"
        os.environ.setdefault(
            "LANGSMITH_PROJECT",
            "firmable-part2-schema-inference",
        )

        return client

    except Exception as e:
        print(
            "LangSmith disabled:",
            f"{type(e).__name__}: {e}",
        )
        os.environ["LANGSMITH_TRACING"] = "false"
        return None


def setup_tavily():
    key = os.getenv("TAVILY_API_KEY")

    if not key or TavilyClient is None:
        return None

    try:
        client = TavilyClient(api_key=key)

        # Validate once. Tavily is Turbo-only.
        client.search(
            query="data.gov.au",
            search_depth="basic",
            max_results=1,
        )

        return client

    except Exception as e:
        print(
            "Tavily disabled:",
            f"{type(e).__name__}: {e}",
        )
        return None


openai_client = setup_openai()
langsmith_client = setup_langsmith()
tavily_client = setup_tavily()

print("OpenAI:", "ready" if openai_client else "disabled")
print("LangSmith:", "enabled" if langsmith_client else "disabled")
print("Tavily:", "enabled for Turbo" if tavily_client else "disabled")

OpenAI: ready
LangSmith: enabled
Tavily: enabled for Turbo


## Helpers


`json_safe` converts nested pandas/NumPy values into native values before JSON writes, review interrupts and checkpoint state updates. Missing values become null, timestamps become ISO strings and numeric/boolean types are preserved. The same conversion handles NumPy arrays and native dates/timedeltas; non-finite numbers become null. JSON writes are strict, without `default=str`.

Gold review shows every schema column and selects enough records from the loaded dataset to cover available values. CSV encoding and parser recovery uses Python while preserving the raw file. Database connections close before human input, on completion and on failed runs; temporary source views are rebuilt on resume.

In [4]:
def json_safe(value):
    """Convert pandas/NumPy values before JSON writes or checkpointing."""
    if value is None:
        return None

    if isinstance(value, dict):
        return {
            str(k): json_safe(v)
            for k, v in value.items()
        }

    if isinstance(value, (list, tuple)):
        return [json_safe(v) for v in value]

    # DuckDB list columns can arrive as NumPy arrays.
    if isinstance(value, np.ndarray):
        return json_safe(value.tolist())

    if isinstance(value, pd.Timestamp):
        return value.isoformat()

    if value is pd.NaT:
        return None

    try:
        if pd.isna(value):
            return None
    except (TypeError, ValueError):
        pass

    # Include native date/time values produced by NumPy .item().
    if isinstance(value, (datetime, date)):
        return value.isoformat()

    if isinstance(value, (pd.Timedelta, timedelta)):
        return pd.Timedelta(value).isoformat()

    # JSON has no Infinity value; handle it like other missing numbers.
    if isinstance(value, (float, np.floating)) and not np.isfinite(value):
        return None

    if isinstance(value, np.generic):
        return json_safe(value.item())

    return value


CTX = {}
ACTIVE_THREAD_ID = None


def reset_context():
    CTX.clear()
    CTX["metric_history"] = []


def dataset_slug(title: str) -> str:
    value = str(title).lower()
    value = re.sub(r"\bdataset\b", "", value)
    value = re.sub(r"[^a-z0-9]+", "_", value)
    return value.strip("_")


def quote_identifier(name: str) -> str:
    return '"' + str(name).replace('"', '""') + '"'


def sql_literal(value: Any) -> str:
    if value is None:
        return "NULL"

    if isinstance(value, bool):
        return "TRUE" if value else "FALSE"

    if isinstance(value, (int, float)) and not isinstance(value, bool):
        return str(value)

    return "'" + str(value).replace("'", "''") + "'"


def message_text(message) -> str:
    if isinstance(message, dict):
        content = message.get("content", "")
    else:
        content = getattr(message, "content", "")

    if isinstance(content, str):
        return content

    if isinstance(content, list):
        parts = []

        for item in content:
            if not isinstance(item, dict):
                continue

            if item.get("type") in {
                "text",
                "output_text",
            }:
                text = item.get("text")

                if text is None:
                    text = item.get("content")

                if text:
                    parts.append(str(text))

        return "\n".join(parts)

    return ""


def extract_json(text: str) -> dict:
    text = text.strip()

    text = re.sub(
        r"^```(?:json)?\s*",
        "",
        text,
        flags=re.IGNORECASE,
    )

    text = re.sub(
        r"\s*```$",
        "",
        text,
    )

    try:
        return json.loads(text)

    except Exception:
        start = text.find("{")
        end = text.rfind("}")

        if start == -1 or end == -1 or end <= start:
            raise ValueError(
                "Agent did not return a JSON object."
            )

        return json.loads(
            text[start:end + 1]
        )


def current_resource():
    resource_id = CTX.get("resource_id")

    for resource in CTX.get("resources", []):
        if resource.get("id") == resource_id:
            return resource

    return None


def candidate_path():
    return REVIEW_DIR / f"{CTX['dataset']['slug']}.candidate.yaml"


def approved_path():
    return CONFIG_DIR / f"{CTX['dataset']['slug']}.mapping.yaml"


def rejected_path():
    return REJECTED_DIR / f"{CTX['dataset']['slug']}.candidate.yaml"


def state_path():
    return STATE_DIR / f"{CTX['dataset']['slug']}.json"


def metrics_path():
    return METRICS_DIR / f"{CTX['dataset']['slug']}.json"


def save_yaml(path: Path, payload: dict):
    path.write_text(
        yaml.safe_dump(
            json_safe(payload),
            sort_keys=False,
            allow_unicode=True,
        ),
        encoding="utf-8",
    )

## Inputs


In [5]:
shortlist = pd.read_csv(PART1_DIR / "ranked_shortlist_50.csv")

with open(ONTOLOGY_PATH, "r", encoding="utf-8") as f:
    ontology = yaml.safe_load(f)

ONTOLOGY_TEXT = yaml.safe_dump(
    json_safe(ontology),
    sort_keys=False,
    allow_unicode=True,
)

FIELD_SPECS = {}
CANONICAL_FIELDS = []

for section in ["entity", "address", "observation"]:
    for field, spec in ontology.get(section, {}).items():
        target = f"{section}.{field}"
        FIELD_SPECS[target] = spec
        CANONICAL_FIELDS.append(target)

VALID_TARGETS = set(CANONICAL_FIELDS)

REQUIRED_OBSERVATION_FIELDS = {
    "observation.source_id",
    "observation.source_record_id",
    "observation.observed_at",
    "observation.ingested_at",
    "observation.licence",
    "observation.extractor_version",
}

print("Part 1 datasets:", len(shortlist))
print("Canonical fields:", len(CANONICAL_FIELDS))

Part 1 datasets: 50
Canonical fields: 23


## Schemas


In [6]:
class ReaderConfig(BaseModel):
    model_config = ConfigDict(extra="forbid")

    format: Literal[
        "csv",
        "tsv",
        "xlsx",
        "xls",
        "json",
        "jsonl",
        "xml",
    ]

    encoding: str | None = None
    delimiter: str | None = None
    header: bool = True
    skip: int = 0
    sheet_name: str | int | None = None
    record_tag: str | None = None
    zip_member: str | None = None


class SourceDescriptor(BaseModel):
    model_config = ConfigDict(extra="forbid")

    dataset_id: str
    dataset_title: str
    resource_id: str
    resource_name: str
    resource_url: str
    resource_format: str
    publisher: str | None = None
    licence: str
    metadata_modified: str | None = None
    resource_last_modified: str | None = None


class Transformation(BaseModel):
    model_config = ConfigDict(extra="forbid")

    type: Literal[
        "direct",
        "sql_expression",
        "constant",
        "record_hash",
    ]

    expression: str | None
    value: str | int | float | bool | None

    @model_validator(mode="after")
    def validate_shape(self):
        if self.type == "sql_expression" and not self.expression:
            raise ValueError(
                "sql_expression requires expression"
            )

        if self.type == "constant" and self.value is None:
            raise ValueError(
                "constant requires value"
            )

        if self.type in {"direct", "record_hash"}:
            if (
                self.expression is not None
                or self.value is not None
            ):
                raise ValueError(
                    f"{self.type} requires "
                    "expression=null and value=null"
                )

        return self


class FieldMapping(BaseModel):
    model_config = ConfigDict(extra="forbid")

    source_fields: list[str]
    target_field: str
    transformation: Transformation

    confidence: float = Field(
        ge=0.0,
        le=1.0,
    )

    derivation_level: Literal[
        "L0",
        "L1",
        "L2",
        "L3",
        "L4",
    ]

    reason: str


class UnmappedField(BaseModel):
    model_config = ConfigDict(extra="forbid")

    source_field: str
    reason: str


class ResearchNote(BaseModel):
    model_config = ConfigDict(extra="forbid")

    finding: str
    source_url: str | None


class AgentMappingOutput(BaseModel):
    model_config = ConfigDict(extra="forbid")

    source_reliability: float = Field(
        ge=0.0,
        le=1.0,
    )

    source_reliability_reason: str

    mappings: list[FieldMapping]
    unmapped_fields: list[UnmappedField]

    research_notes: list[ResearchNote]
    agent_notes: list[str]


class MappingConfig(BaseModel):
    model_config = ConfigDict(extra="forbid")

    config_version: str = "1.0"

    source: SourceDescriptor
    reader: ReaderConfig

    source_reliability: float
    source_reliability_reason: str

    mappings: list[FieldMapping]

    runtime_fields: dict[str, str]

    unmapped_fields: list[UnmappedField]

    research_notes: list[ResearchNote] = Field(default_factory=list)
    agent_notes: list[str] = Field(default_factory=list)


class ResourceChoice(BaseModel):
    model_config = ConfigDict(extra="forbid")

    resource_id: str
    reason: str


class ReaderDecision(BaseModel):
    model_config = ConfigDict(extra="forbid")

    file_path: str
    delimiter: str | None
    header: bool
    skip: int
    sheet_name: str | int | None
    record_tag: str | None
    reason: str

    @property
    def options(self) -> dict[str, Any]:
        options = {
            "header": self.header,
            "skip": self.skip,
        }

        if self.delimiter is not None:
            options["delimiter"] = self.delimiter

        if self.sheet_name is not None:
            options["sheet_name"] = self.sheet_name

        if self.record_tag is not None:
            options["record_tag"] = self.record_tag

        return options

## Discovery tools


In [7]:
@tool
def find_dataset(dataset_name: str) -> str:
    """Find one Part 1 dataset by title or ID and set it as active."""

    titles = shortlist["title"].fillna("").astype(str)

    exact = shortlist[
        shortlist["dataset_id"].astype(str) == dataset_name
    ]
    if exact.empty:
        exact = shortlist[
            titles.str.casefold() == dataset_name.casefold()
        ]

    if len(exact) == 1:
        row = exact.iloc[0]

    else:
        contains = shortlist[
            titles.str.casefold().str.contains(
                re.escape(dataset_name.casefold()),
                regex=True,
            )
        ]

        if len(contains) != 1:
            return json.dumps(json_safe({
                "found": False,
                "query": dataset_name,
                "suggestions": contains[
                    "title"
                ].head(10).tolist(),
            }), allow_nan=False)

        row = contains.iloc[0]

    CTX["dataset"] = json_safe({
        "dataset_id": row["dataset_id"],
        "title": row["title"],
        "slug": dataset_slug(row["title"]),
        "publisher": row.get(
            "publishing_organisation"
        ),
        "part1_confidence": row.get(
            "confidence"
        ),
    })

    return json.dumps(
        json_safe({
            "found": True,
            **CTX["dataset"],
        }),
        indent=2,
        allow_nan=False,
    )


@tool
def get_dataset_metadata() -> str:
    """Retrieve CKAN metadata for the active dataset."""

    if "dataset" not in CTX:
        return "ERROR: call find_dataset first."

    response = requests.get(
        f"{CKAN_API}/package_show",
        params={
            "id": CTX["dataset"]["dataset_id"]
        },
        timeout=60,
    )

    response.raise_for_status()

    payload = response.json()

    if not payload.get("success"):
        return "ERROR: CKAN package_show failed."

    metadata = payload["result"]

    CTX["metadata"] = metadata
    CTX["resources"] = metadata.get(
        "resources",
        [],
    )

    return json.dumps(json_safe({
        "dataset_id": metadata.get("id"),
        "title": metadata.get("title"),
        "publisher": (
            metadata.get("organization") or {}
        ).get("title"),
        "description": metadata.get("notes"),
        "licence": metadata.get(
            "license_title"
        ),
        "metadata_modified": metadata.get(
            "metadata_modified"
        ),
        "resource_count": len(
            CTX["resources"]
        ),
    }), allow_nan=False, indent=2, ensure_ascii=False)


@tool
def list_resources() -> str:
    """List CKAN resources so the agent can choose the actual data resource."""

    if "metadata" not in CTX:
        result = get_dataset_metadata.invoke({})

        if str(result).startswith("ERROR"):
            return str(result)

    resources = []

    for item in CTX.get("resources", []):
        resources.append({
            "resource_id": item.get("id"),
            "name": item.get("name"),
            "description": item.get(
                "description"
            ),
            "format": item.get("format"),
            "url": item.get("url"),
            "size": item.get("size"),
            "last_modified": item.get(
                "last_modified"
            ),
        })

    return json.dumps(
        json_safe(resources),
        indent=2,
        ensure_ascii=False,
        allow_nan=False,
    )

## Resource tools


In [8]:
def _resource_filename(resource: dict) -> str:
    url_path = Path(
        urlparse(
            resource.get("url") or ""
        ).path
    ).name

    if url_path and "." in url_path:
        return url_path

    name = (
        resource.get("name")
        or resource.get("id")
        or "resource"
    )

    safe = re.sub(
        r"[^A-Za-z0-9._-]+",
        "_",
        name,
    ).strip("_")

    fmt = (
        str(
            resource.get("format")
            or ""
        )
        .strip()
        .lower()
        .lstrip(".")
    )

    if fmt and not safe.lower().endswith(
        f".{fmt}"
    ):
        safe = f"{safe}.{fmt}"

    return safe


@tool
def download_resource(
    resource_id: str,
) -> str:
    """Download the resource selected by the agent and extract ZIP files."""

    if "dataset" not in CTX:
        return "ERROR: call find_dataset first."

    if (
        CTX.get("source_table")
        and CTX.get("resource_id")
        and CTX.get("resource_id") != resource_id
    ):
        return (
            "ERROR: a source resource is already loaded. "
            "Normal onboarding must not switch resources after profiling. "
            "Use the human/Turbo path if the selected source is genuinely wrong."
        )

    if "resources" not in CTX:
        list_resources.invoke({})

    matches = [
        item
        for item in CTX.get(
            "resources",
            [],
        )
        if item.get("id")
        == resource_id
    ]

    if len(matches) != 1:
        return (
            "ERROR: resource_id not found: "
            + resource_id
        )

    resource = matches[0]

    url = resource.get("url")

    if not url:
        return (
            "ERROR: selected resource "
            "has no URL."
        )

    dataset_dir = (
        RAW_DIR
        / CTX["dataset"]["slug"]
    )

    dataset_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    file_path = (
        dataset_dir
        / _resource_filename(resource)
    )

    if (
        not file_path.exists()
        or file_path.stat().st_size == 0
    ):
        with requests.get(
            url,
            stream=True,
            timeout=(30, 300),
            allow_redirects=True,
        ) as response:

            response.raise_for_status()

            with open(
                file_path,
                "wb",
            ) as f:

                for chunk in response.iter_content(
                    chunk_size=1024 * 1024
                ):
                    if chunk:
                        f.write(chunk)

    files = [file_path]

    if file_path.suffix.lower() == ".zip":
        extract_dir = (
            dataset_dir
            / "extracted"
        )

        extract_dir.mkdir(
            parents=True,
            exist_ok=True,
        )

        with zipfile.ZipFile(
            file_path
        ) as archive:
            archive.extractall(
                extract_dir
            )

        files = [
            p
            for p in extract_dir.rglob(
                "*"
            )
            if p.is_file()
        ]

    CTX["resource_id"] = resource_id
    CTX["downloaded_path"] = str(
        file_path
    )
    CTX["downloaded_files"] = [
        str(p)
        for p in files
    ]

    return json.dumps(json_safe({
        "resource_id": resource_id,
        "downloaded": str(file_path),
        "files": [
            str(p)
            for p in files
        ],
    }), allow_nan=False, indent=2)


@tool
def inspect_file(
    file_path: str,
) -> str:
    """Inspect a downloaded file before loading it."""

    path = Path(file_path)

    if not path.exists():
        return (
            "ERROR: file does not exist: "
            + file_path
        )

    ext = path.suffix.lower()

    result = {
        "file_path": str(path),
        "extension": ext,
        "size_mb": round(
            path.stat().st_size
            / 1024
            / 1024,
            3,
        ),
    }

    try:
        if ext in {
            ".csv",
            ".tsv",
            ".txt",
        }:

            sep = (
                "\t"
                if ext == ".tsv"
                else None
            )

            prepared = prepare_csv_input(
                path, STATE_DIR / "reader_cache" / CTX.get("dataset", {}).get("slug", "inspection")
            )
            result["encoding"] = prepared["encoding"]
            df, preview_info = read_csv_python(
                prepared["file_path"], delimiter=sep, limit=8
            )
            result["reader_recovery"] = preview_info

            result["columns"] = (
                df.columns.tolist()
            )

            result["sample"] = (
                json_safe(df.to_dict("records"))
            )

        elif ext in {
            ".xlsx",
            ".xls",
        }:

            xls = pd.ExcelFile(path)

            result["sheets"] = (
                xls.sheet_names
            )

            previews = {}

            for sheet in xls.sheet_names[:5]:
                try:
                    raw = pd.read_excel(
                        path,
                        sheet_name=sheet,
                        header=None,
                        nrows=12,
                        dtype=str,
                    )

                    previews[sheet] = (
                        json_safe(raw.to_dict("records"))
                    )

                except Exception as e:
                    previews[sheet] = {
                        "error": str(e)
                    }

            result[
                "sheet_previews"
            ] = previews

        elif ext in {
            ".json",
            ".jsonl",
            ".ndjson",
        }:

            df = pd.read_json(
                path,
                lines=(
                    ext
                    in {
                        ".jsonl",
                        ".ndjson",
                    }
                ),
            ).head(8)

            result["columns"] = (
                df.columns.tolist()
            )

            result["sample"] = (
                json_safe(df.to_dict("records"))
            )

        elif ext == ".xml":

            counts = Counter()
            seen = 0

            for _, elem in ET.iterparse(
                path,
                events=("end",),
            ):
                tag = elem.tag.split(
                    "}"
                )[-1]

                counts[tag] += 1
                seen += 1

                elem.clear()

                if seen >= 10000:
                    break

            result["common_tags"] = (
                counts.most_common(30)
            )

            result["hint"] = (
                "Choose the repeating record "
                "tag and pass it as record_tag "
                "to load_source."
            )

        else:
            result["hint"] = (
                "Use Turbo sandbox inspection "
                "if this format needs custom "
                "handling."
            )

    except Exception as e:
        result["inspection_error"] = (
            f"{type(e).__name__}: {e}"
        )

    return json.dumps(
        json_safe(result),
        indent=2,
        ensure_ascii=False,
        allow_nan=False,
    )

## Source tools


In [9]:
def _flatten_xml_record(element):
    values = {}

    def add_value(key, value):
        if value is None:
            return

        value = str(value).strip()

        if not value:
            return

        if key not in values:
            values[key] = value
            return

        existing = values[key]

        if not isinstance(existing, list):
            existing = [existing]

        existing.append(value)
        values[key] = existing

    def walk(node, prefix=""):
        tag = node.tag.split("}")[-1]
        path = f"{prefix}.{tag}" if prefix else tag

        for key, value in node.attrib.items():
            add_value(
                f"{path}.@{key}",
                value,
            )

        children = list(node)
        text = (node.text or "").strip()

        if not children:
            add_value(path, text)

        for child in children:
            walk(child, path)

    walk(element)

    result = {}

    for key, value in values.items():
        result[key] = (
            json.dumps(
                json_safe(value), allow_nan=False,
                ensure_ascii=False,
            )
            if isinstance(value, list)
            else value
        )

    return result



def _xml_to_nested(element):
    obj = {}

    for key, value in element.attrib.items():
        obj[f"@{key}"] = value

    children = list(element)
    text = (element.text or "").strip()

    if not children:
        if obj:
            if text:
                obj["#text"] = text
            return obj
        return text or None

    grouped = {}

    for child in children:
        tag = child.tag.split("}")[-1]
        grouped.setdefault(tag, []).append(
            _xml_to_nested(child)
        )

    for tag, values in grouped.items():
        obj[tag] = values[0] if len(values) == 1 else values

    if text:
        obj["#text"] = text

    return obj
def _xml_to_jsonl(
    xml_path: Path,
    record_tag: str,
) -> Path:

    output_path = (
        xml_path.parent
        / (
            xml_path.stem
            + f".{record_tag}.jsonl"
        )
    )

    with open(
        output_path,
        "w",
        encoding="utf-8",
    ) as out:

        for _, elem in ET.iterparse(
            xml_path,
            events=("end",),
        ):

            tag = elem.tag.split("}")[-1]

            if tag != record_tag:
                continue

            record = _flatten_xml_record(
                elem
            )

            # Preserve the original source record structure as JSON too.
            record["raw_json"] = json.dumps(
                json_safe({
                    tag: _xml_to_nested(elem)
                }),
                ensure_ascii=False,
                allow_nan=False,
            )

            out.write(
                json.dumps(
                    json_safe(record),
                    ensure_ascii=False,
                    allow_nan=False,
                )
                + "\n"
            )

            elem.clear()

    return output_path
def _reader_config(
    path: Path,
    options: dict,
) -> ReaderConfig:

    ext = path.suffix.lower()

    fmt = {
        ".csv": "csv",
        ".tsv": "tsv",
        ".txt": "csv",
        ".xlsx": "xlsx",
        ".xls": "xls",
        ".json": "json",
        ".jsonl": "jsonl",
        ".ndjson": "jsonl",
        ".xml": "xml",
    }.get(ext)

    if not fmt:
        raise ValueError(
            f"Unsupported source format: {ext}"
        )

    zip_member = None

    downloaded_path = Path(
        CTX.get(
            "downloaded_path",
            path,
        )
    )

    if (
        downloaded_path.suffix.lower()
        == ".zip"
        and path != downloaded_path
    ):
        try:
            zip_member = str(
                path.relative_to(
                    downloaded_path.parent
                    / "extracted"
                )
            )
        except Exception:
            zip_member = path.name

    return ReaderConfig(
        format=fmt,
        encoding=options.get("encoding"),
        delimiter=options.get(
            "delimiter"
        ),
        header=options.get(
            "header",
            True,
        ),
        skip=int(
            options.get(
                "skip",
                0,
            )
            or 0
        ),
        sheet_name=options.get(
            "sheet_name"
        ),
        record_tag=options.get(
            "record_tag"
        ),
        zip_member=zip_member,
    )


def _prepare_source_views(
    source_table: str,
):
    with duckdb_lock:
        con.execute(
            "DROP TABLE IF EXISTS _firmable_working"
        )

        con.execute(
            "DROP TABLE IF EXISTS validation_sample"
        )

        con.execute(
            "DROP TABLE IF EXISTS source_data"
        )

        con.execute(f"""
            CREATE TEMP TABLE _firmable_working AS
            SELECT
                row_number() OVER () AS __firmable_row_id,
                *
            FROM {quote_identifier(source_table)}
        """)

        total = con.execute(
            """
            SELECT COUNT(*)
            FROM _firmable_working
            """
        ).fetchone()[0]

        holdout = min(
            VALIDATION_ROWS,
            max(
                1,
                total // 5,
            ),
        )

        con.execute(f"""
            CREATE TEMP TABLE validation_sample AS
            SELECT *
            FROM _firmable_working
            ORDER BY hash(
                __firmable_row_id
                + {HOLDOUT_SEED}
            )
            LIMIT {holdout}
        """)

        con.execute("""
            CREATE TEMP TABLE source_data AS
            SELECT w.*
            FROM _firmable_working w
            LEFT JOIN validation_sample v
                USING (__firmable_row_id)
            WHERE
                v.__firmable_row_id
                IS NULL
        """)

        columns = [
            row[0]
            for row in con.execute(
                "DESCRIBE source_data"
            ).fetchall()
            if row[0]
            != "__firmable_row_id"
        ]

    CTX["source_table"] = source_table
    CTX["source_columns"] = columns
    CTX["source_rows"] = total
    CTX["validation_rows"] = holdout


def _load_source_file(
    file_path: str,
    options: dict,
) -> dict:

    if "dataset" not in CTX:
        raise RuntimeError(
            "find_dataset must run first"
        )

    path = Path(file_path)

    if not path.exists():
        raise FileNotFoundError(
            file_path
        )

    slug = CTX["dataset"]["slug"]
    source_table = f"src_{slug}"

    options = dict(options)
    reader_info = None
    read_path = path
    if path.suffix.lower() in {".csv", ".tsv", ".txt"}:
        reader_info = prepare_csv_input(
            path, STATE_DIR / "reader_cache" / slug, options.get("encoding")
        )
        read_path = Path(reader_info["file_path"])
        options["encoding"] = reader_info["encoding"]
        if reader_info["converted_to_utf8"]:
            print(f"PART 2 — Reader recovery: {path.name}; {reader_info['encoding']} → UTF-8; original file preserved.")

    reader = _reader_config(
        path,
        options,
    )

    path_sql = str(
        read_path
    ).replace(
        "'",
        "''",
    )

    with duckdb_lock:

        if reader.format in {
            "csv",
            "tsv",
        }:

            args = [
                "sample_size=-1",
                "all_varchar=true",
                (
                    "header=true"
                    if reader.header
                    else "header=false"
                ),
            ]

            delimiter = (
                reader.delimiter
                or (
                    "\t"
                    if reader.format
                    == "tsv"
                    else None
                )
            )

            if delimiter is not None:
                args.append(
                    "delim="
                    + sql_literal(
                        delimiter
                    )
                )

            if reader.skip:
                args.append(
                    f"skip={reader.skip}"
                )

            try:
                con.execute(f"""
                    CREATE OR REPLACE TABLE {quote_identifier(source_table)} AS
                    SELECT * FROM read_csv_auto('{path_sql}', {", ".join(args)})
                """)
            except duckdb.Error:
                print(f"PART 2 — Reader recovery: retry {path.name} with Python's CSV parser.")
                df, recovery = read_csv_python(
                    read_path, delimiter=delimiter, header=reader.header, skip=reader.skip
                )
                reader_info.update(recovery)
                df.columns = [str(column) for column in df.columns]
                temp_name = "_csv_" + uuid.uuid4().hex
                con.register(temp_name, df)
                try:
                    con.execute(f"CREATE OR REPLACE TABLE {quote_identifier(source_table)} AS SELECT * FROM {quote_identifier(temp_name)}")
                finally:
                    con.unregister(temp_name)
                print(f"PART 2 — Python reader loaded {len(df):,} records; all rows retained.")


        elif reader.format in {
            "xlsx",
            "xls",
        }:

            sheet_name = (
                reader.sheet_name
                if reader.sheet_name
                is not None
                else 0
            )

            header_row = (
                reader.skip
                if reader.header
                else None
            )

            df = pd.read_excel(
                path,
                sheet_name=sheet_name,
                header=header_row,
                dtype=str,
            )

            temp_name = (
                "_excel_"
                + uuid.uuid4().hex
            )

            con.register(
                temp_name,
                df,
            )

            con.execute(f"""
                CREATE OR REPLACE TABLE
                    {quote_identifier(source_table)}
                AS
                SELECT *
                FROM {quote_identifier(temp_name)}
            """)

            con.unregister(
                temp_name
            )

        elif reader.format in {
            "json",
            "jsonl",
        }:

            con.execute(f"""
                CREATE OR REPLACE TABLE
                    {quote_identifier(source_table)}
                AS
                SELECT *
                FROM read_json_auto(
                    '{path_sql}'
                )
            """)

        elif reader.format == "xml":

            if not reader.record_tag:
                raise ValueError(
                    "XML requires record_tag"
                )

            jsonl_path = _xml_to_jsonl(
                path,
                reader.record_tag,
            )

            jsonl_sql = str(
                jsonl_path
            ).replace(
                "'",
                "''",
            )

            con.execute(f"""
                CREATE OR REPLACE TABLE
                    {quote_identifier(source_table)}
                AS
                SELECT *
                FROM read_json_auto(
                    '{jsonl_sql}'
                )
            """)

        else:
            raise ValueError(
                "Unsupported reader format: "
                + reader.format
            )

    CTX["file_path"] = str(path)
    CTX["reader"] = reader
    CTX["load_options"] = options
    CTX["reader_recovery"] = reader_info

    _prepare_source_views(
        source_table
    )

    return json_safe({
        "source_table": source_table,
        "file_path": str(path),
        "reader": reader.model_dump(),
        "reader_recovery": reader_info,
        "rows": CTX["source_rows"],
        "validation_rows": (
            CTX["validation_rows"]
        ),
        "columns": (
            CTX["source_columns"]
        ),
    })


@tool
def load_source(
    file_path: str,
    options_json: str = "{}",
) -> str:
    """Load the selected file into src_<dataset> and create a held-out sample."""

    try:
        options = json.loads(
            options_json
            or "{}"
        )

        result = _load_source_file(
            file_path,
            options,
        )

        return json.dumps(
            json_safe(result),
            indent=2,
            allow_nan=False,
        )

    except Exception as e:
        return (
            "ERROR: "
            f"{type(e).__name__}: {e}"
        )

## Profile tools


In [10]:
@tool
def profile_source() -> str:
    """Profile source_data and save a reusable dataset profile."""

    if "source_table" not in CTX:
        return (
            "ERROR: call load_source first."
        )

    with duckdb_lock:
        schema_rows = con.execute(
            "DESCRIBE source_data"
        ).fetchall()

        sample_df = con.execute("""
            SELECT *
            EXCLUDE (__firmable_row_id)
            FROM source_data
            LIMIT 10
        """).df()

        stats_df = con.execute("""
            SELECT *
            EXCLUDE (__firmable_row_id)
            FROM source_data
            USING SAMPLE 5000 ROWS
        """).df()

        agent_rows = con.execute(
            """
            SELECT COUNT(*)
            FROM source_data
            """
        ).fetchone()[0]

    columns = []

    for name, dtype, *_ in schema_rows:
        if name == "__firmable_row_id":
            continue

        series = (
            stats_df[name]
            if name in stats_df.columns
            else pd.Series(
                dtype=object
            )
        )

        non_null = (
            series.dropna()
        )

        columns.append({
            "name": name,
            "dtype": dtype,
            "sample_null_rate": (
                round(
                    float(
                        series.isna().mean()
                    ),
                    4,
                )
                if len(series)
                else None
            ),
            "sample_distinct": (
                int(
                    non_null.nunique(
                        dropna=True
                    )
                )
                if len(series)
                else 0
            ),
            "examples": [
                str(x)[:300]
                for x in (
                    non_null
                    .astype(str)
                    .drop_duplicates()
                    .head(5)
                    .tolist()
                )
            ],
        })

    metadata = CTX.get(
        "metadata",
        {},
    )

    resource = (
        current_resource()
        or {}
    )

    profile = {
        "metadata": {
            "dataset_id": (
                CTX["dataset"][
                    "dataset_id"
                ]
            ),
            "title": (
                CTX["dataset"][
                    "title"
                ]
            ),
            "publisher": (
                CTX["dataset"].get(
                    "publisher"
                )
            ),
            "resource_id": (
                CTX.get(
                    "resource_id"
                )
            ),
            "resource_name": (
                resource.get(
                    "name"
                )
            ),
            "resource_format": (
                resource.get(
                    "format"
                )
            ),
            "licence": (
                metadata.get(
                    "license_title"
                )
            ),
            "metadata_modified": (
                metadata.get(
                    "metadata_modified"
                )
            ),
            "file_path": (
                CTX.get(
                    "file_path"
                )
            ),
            "reader": (
                CTX["reader"]
                .model_dump()
            ),
        },
        "dataset_stats": {
            "row_count": (
                CTX.get(
                    "source_rows"
                )
            ),
            "agent_rows": (
                agent_rows
            ),
            "validation_rows": (
                CTX.get(
                    "validation_rows"
                )
            ),
            "column_count": (
                len(columns)
            ),
        },
        "columns": columns,
        "sample_rows": (
            json_safe(sample_df.to_dict("records"))
        ),
    }

    profile = json_safe(profile)

    path = (
        PROFILE_DIR
        / (
            CTX["dataset"]["slug"]
            + ".profile.json"
        )
    )

    path.write_text(
        json.dumps(
            json_safe(profile),
            indent=2,
            ensure_ascii=False,
            allow_nan=False,
        ),
        encoding="utf-8",
    )

    CTX["profile"] = profile
    CTX["profile_path"] = str(
        path
    )

    return json.dumps(
        json_safe(profile),
        indent=2,
        ensure_ascii=False,
        allow_nan=False,
    )


BLOCKED_SQL = re.compile(
    r"\b("
    r"insert|update|delete|drop|alter|"
    r"create|copy|attach|detach|install|"
    r"load|pragma|validation_sample|"
    r"_firmable_working|information_schema|"
    r"duckdb_"
    r")\b",
    re.IGNORECASE,
)


@tool
def query_source(
    sql: str,
) -> str:
    """Run read-only DuckDB SQL against source_data only."""

    sql = sql.strip().rstrip(
        ";"
    )

    if not sql.lower().startswith(
        (
            "select",
            "with",
        )
    ):
        return (
            "ERROR: query must start "
            "with SELECT or WITH."
        )

    if BLOCKED_SQL.search(sql):
        return (
            "ERROR: blocked SQL."
        )

    if not re.search(
        r"\bsource_data\b",
        sql,
        re.IGNORECASE,
    ):
        return (
            "ERROR: query must use "
            "source_data."
        )

    try:
        limited_sql = f"""
            SELECT *
            FROM ({sql}) AS agent_query
            LIMIT 100
        """

        with duckdb_lock:
            df = con.execute(
                limited_sql
            ).df()

        return json.dumps(
            json_safe(df.to_dict("records")),
            indent=2,
            ensure_ascii=False,
            allow_nan=False,
        )

    except Exception as e:
        return (
            "ERROR: "
            f"{type(e).__name__}: {e}"
        )


def deterministic_identifier_checks() -> dict:
    """
    Check likely ABN/ACN/NZBN columns deterministically on source_data.
    This is host validation, not model-generated SQL.
    """

    checks = {}

    columns = CTX.get(
        "source_columns",
        [],
    )

    for column in columns:
        normalized = re.sub(
            r"[^a-z0-9]+",
            "",
            str(column).lower(),
        )

        identifier_type = None

        if (
            normalized == "abn"
            or "australianbusinessnumber" in normalized
        ):
            identifier_type = "abn"

        elif (
            normalized == "acn"
            or "australiancompanynumber" in normalized
        ):
            identifier_type = "acn"

        elif normalized == "nzbn":
            identifier_type = "nzbn"

        if identifier_type is None:
            continue

        try:
            with duckdb_lock:
                df = con.execute(
                    f"""
                    SELECT {quote_identifier(column)} AS value
                    FROM source_data
                    WHERE {quote_identifier(column)} IS NOT NULL
                    LIMIT 5000
                    """
                ).df()

            values = [
                str(value).strip()
                for value in df["value"].tolist()
                if not is_null(value)
                and str(value).strip()
            ]

            if identifier_type == "abn":
                valid = sum(
                    valid_abn(value)
                    for value in values
                )

            elif identifier_type == "acn":
                valid = sum(
                    valid_acn(value)
                    for value in values
                )

            else:
                valid = sum(
                    bool(
                        re.fullmatch(
                            r"\d{13}",
                            value,
                        )
                    )
                    for value in values
                )

            checks[column] = {
                "identifier_type": identifier_type,
                "sample_non_null": len(values),
                "valid_count": int(valid),
                "valid_rate": (
                    round(
                        valid / len(values),
                        4,
                    )
                    if values
                    else None
                ),
                "examples": values[:5],
            }

        except Exception as e:
            checks[column] = {
                "identifier_type": identifier_type,
                "error": (
                    f"{type(e).__name__}: {e}"
                ),
            }

    return json_safe(checks)

## Turbo tools


In [11]:
@tool
def run_python(
    code: str,
) -> str:
    """Run Python in a restricted project sandbox for deeper testing."""

    if "dataset" not in CTX:
        return (
            "ERROR: no active dataset."
        )

    workdir = (
        SANDBOX_DIR
        / CTX["dataset"]["slug"]
    )

    workdir.mkdir(
        parents=True,
        exist_ok=True,
    )

    safe_env = {
        "PATH": os.environ.get(
            "PATH",
            "",
        ),
        "PYTHONPATH": os.environ.get(
            "PYTHONPATH",
            "",
        ),
        "FIRMABLE_PROJECT_ROOT": str(
            PROJECT_ROOT
        ),
        "FIRMABLE_DUCKDB": str(
            DB_PATH
        ),
        "FIRMABLE_DATA_FILE": str(
            CTX.get(
                "file_path"
            )
            or ""
        ),
        "FIRMABLE_SOURCE_TABLE": str(
            CTX.get(
                "source_table"
            )
            or ""
        ),
    }

    try:
        result = subprocess.run(
            [
                sys.executable,
                "-c",
                code,
            ],
            cwd=workdir,
            env=safe_env,
            capture_output=True,
            text=True,
            timeout=120,
        )

        return json.dumps(json_safe({
            "return_code": (
                result.returncode
            ),
            "stdout": (
                result.stdout[
                    -12000:
                ]
            ),
            "stderr": (
                result.stderr[
                    -12000:
                ]
            ),
            "sandbox": str(
                workdir
            ),
        }), allow_nan=False, indent=2)

    except Exception as e:
        return json.dumps(json_safe({
            "return_code": -1,
            "stdout": "",
            "stderr": (
                f"{type(e).__name__}: {e}"
            ),
            "sandbox": str(
                workdir
            ),
        }), allow_nan=False, indent=2)


@tool
def search_docs(
    query: str,
    domains_csv: str = "",
) -> str:
    """Search authoritative documentation. Turbo only."""

    if tavily_client is None:
        return (
            "ERROR: Tavily is unavailable. "
            "Continue without web research."
        )

    domains = [
        item.strip()
        for item in domains_csv.split(",")
        if item.strip()
    ][:5]

    kwargs = {
        "query": query,
        "search_depth": "advanced",
        "max_results": 4,
    }

    if domains:
        kwargs[
            "include_domains"
        ] = domains

    try:
        result = (
            tavily_client.search(
                **kwargs
            )
        )

        compact = []

        for item in result.get(
            "results",
            [],
        ):
            compact.append({
                "title": item.get(
                    "title"
                ),
                "url": item.get(
                    "url"
                ),
                "content": (
                    item.get(
                        "content"
                    )
                    or ""
                )[:1500],
            })

        return json.dumps(
            json_safe(compact), allow_nan=False,
            indent=2,
            ensure_ascii=False,
        )

    except Exception as e:
        return (
            "ERROR: Tavily failed: "
            f"{type(e).__name__}: {e}"
        )

## Validation


In [12]:
def is_null(value):
    if value is None:
        return True

    try:
        return bool(
            pd.isna(value)
        )

    except Exception:
        return False


def valid_abn(value):
    if is_null(value):
        return False

    value = str(
        value
    ).strip()

    if not re.fullmatch(
        r"\d{11}",
        value,
    ):
        return False

    digits = [
        int(x)
        for x in value
    ]

    digits[0] -= 1

    weights = [
        10,
        1,
        3,
        5,
        7,
        9,
        11,
        13,
        15,
        17,
        19,
    ]

    return (
        sum(
            d * w
            for d, w
            in zip(
                digits,
                weights,
            )
        )
        % 89
        == 0
    )


def valid_acn(value):
    if is_null(value):
        return False

    value = str(
        value
    ).strip()

    if not re.fullmatch(
        r"\d{9}",
        value,
    ):
        return False

    digits = [
        int(x)
        for x in value
    ]

    weights = [
        8,
        7,
        6,
        5,
        4,
        3,
        2,
        1,
    ]

    total = sum(
        digits[i]
        * weights[i]
        for i in range(8)
    )

    check_digit = (
        10 - total % 10
    ) % 10

    return (
        digits[8]
        == check_digit
    )


def canonical_value_valid(
    target,
    value,
):
    if is_null(value):
        return True

    text = str(
        value
    ).strip()

    if not text:
        return True

    if target == "entity.abn":
        return valid_abn(text)

    if target == "entity.acn":
        return valid_acn(text)

    if target == "entity.nzbn":
        return bool(
            re.fullmatch(
                r"\d{13}",
                text,
            )
        )

    spec = FIELD_SPECS.get(
        target,
        {},
    )

    if isinstance(
        spec,
        dict,
    ):
        allowed = (
            spec.get("values")
            or spec.get("enum")
        )

        if allowed:
            return text in allowed

        pattern = spec.get(
            "pattern"
        )

        if (
            pattern
            and not re.fullmatch(
                pattern,
                text,
            )
        ):
            return False

        field_type = spec.get(
            "type"
        )

        if field_type in {
            "date",
            "datetime",
        }:
            return not pd.isna(
                pd.to_datetime(
                    value,
                    errors="coerce",
                )
            )

    return True


UNSAFE_EXPRESSION = re.compile(
    r"(;|\bselect\b|\bfrom\b|"
    r"\bcopy\b|\battach\b|"
    r"\bdetach\b|\bpragma\b|"
    r"\binstall\b|\bload\b|"
    r"\bread_[a-z_]*\s*\()",
    re.IGNORECASE,
)


def source_descriptor():
    resource = (
        current_resource()
        or {}
    )

    metadata = CTX.get(
        "metadata",
        {},
    )

    organisation = (
        metadata.get(
            "organization"
        )
        or {}
    )

    return SourceDescriptor(
        dataset_id=(
            CTX["dataset"][
                "dataset_id"
            ]
        ),
        dataset_title=(
            CTX["dataset"][
                "title"
            ]
        ),
        resource_id=(
            CTX["resource_id"]
        ),
        resource_name=(
            resource.get(
                "name"
            )
            or ""
        ),
        resource_url=(
            resource.get(
                "url"
            )
            or ""
        ),
        resource_format=(
            resource.get(
                "format"
            )
            or ""
        ),
        publisher=(
            organisation.get(
                "title"
            )
            or CTX["dataset"].get(
                "publisher"
            )
        ),
        licence=(
            metadata.get(
                "license_title"
            )
            or metadata.get(
                "license_id"
            )
            or "unknown"
        ),
        metadata_modified=(
            metadata.get(
                "metadata_modified"
            )
        ),
        resource_last_modified=(
            resource.get(
                "last_modified"
            )
        ),
    )


def add_system_envelope(
    agent_output: AgentMappingOutput,
) -> MappingConfig:

    source = source_descriptor()

    mappings = [
        mapping
        for mapping
        in agent_output.mappings
        if mapping.target_field
        not in {
            "observation.source_id",
            "observation.licence",
            "observation.observed_at",
            "observation.ingested_at",
            "observation.extractor_version",
        }
    ]

    mappings += [
        FieldMapping(
            source_fields=[],
            target_field=(
                "observation.source_id"
            ),
            transformation=Transformation(
                type="constant",
                expression=None,
                value=source.resource_id,
            ),
            confidence=1.0,
            derivation_level="L0",
            reason=(
                "Deterministic CKAN "
                "resource identifier."
            ),
        ),
        FieldMapping(
            source_fields=[],
            target_field=(
                "observation.licence"
            ),
            transformation=Transformation(
                type="constant",
                expression=None,
                value=source.licence,
            ),
            confidence=1.0,
            derivation_level="L0",
            reason=(
                "Deterministic CKAN "
                "dataset licence."
            ),
        ),
    ]

    runtime_fields = {
        "observation.observed_at":
            "resource_timestamp_or_ingestion_time",
        "observation.ingested_at":
            "ingestion_time",
        "observation.extractor_version":
            "extractor_version",
    }

    return MappingConfig(
        source=source,
        reader=CTX["reader"],
        source_reliability=(
            agent_output.source_reliability
        ),
        source_reliability_reason=(
            agent_output
            .source_reliability_reason
        ),
        mappings=mappings,
        runtime_fields=runtime_fields,
        unmapped_fields=(
            agent_output.unmapped_fields
        ),
        research_notes=(
            agent_output.research_notes
        ),
        agent_notes=(
            agent_output.agent_notes
        ),
    )


def _mapping_values(
    mapping: FieldMapping,
    relation: str,
) -> pd.Series:

    transform = (
        mapping.transformation
    )

    with duckdb_lock:
        count = con.execute(
            f"""
            SELECT COUNT(*)
            FROM {relation}
            """
        ).fetchone()[0]

    if transform.type == "constant":
        return pd.Series(
            [transform.value] * count
        )

    if transform.type == "direct":
        if len(
            mapping.source_fields
        ) != 1:
            raise ValueError(
                "direct requires one "
                "source field"
            )

        expression = quote_identifier(
            mapping.source_fields[0]
        )

    elif transform.type == "sql_expression":
        expression = (
            transform.expression
            or ""
        )

        if UNSAFE_EXPRESSION.search(
            expression
        ):
            raise ValueError(
                "unsafe SQL expression"
            )

    elif transform.type == "record_hash":
        if not mapping.source_fields:
            raise ValueError(
                "record_hash requires "
                "source_fields"
            )

        pieces = [
            (
                "COALESCE("
                f"CAST({quote_identifier(field)} "
                "AS VARCHAR), '∅')"
            )
            for field
            in mapping.source_fields
        ]

        expression = (
            "sha256(concat_ws('||', "
            + ", ".join(pieces)
            + "))"
        )

    else:
        raise ValueError(
            "Unsupported transform: "
            + transform.type
        )

    with duckdb_lock:
        return con.execute(f"""
            SELECT
                {expression}
                AS __firmable_value
            FROM {relation}
        """).df()[
            "__firmable_value"
        ]


def validate_config(
    config: MappingConfig,
) -> dict:

    errors = []
    warnings = []
    metrics = {}

    source_columns = set(
        CTX.get(
            "source_columns",
            [],
        )
    )

    targets = {}
    semantic_sources = set()

    for i, mapping in enumerate(
        config.mappings
    ):
        prefix = f"mapping[{i}]"
        target = mapping.target_field

        targets.setdefault(
            target,
            [],
        ).append(mapping)

        if target not in VALID_TARGETS:
            errors.append(
                f"{prefix}: unknown target {target}"
            )
            continue

        for field in mapping.source_fields:
            if field not in source_columns:
                errors.append(
                    f"{prefix}: unknown "
                    f"source field {field}"
                )

        if target != (
            "observation.source_record_id"
        ):
            semantic_sources.update(
                mapping.source_fields
            )

        if (
            mapping.transformation.type
            == "direct"
            and len(
                mapping.source_fields
            ) != 1
        ):
            errors.append(
                f"{prefix}: direct requires "
                "one source field"
            )
            continue

        if (
            mapping.transformation.type
            == "record_hash"
            and not mapping.source_fields
        ):
            errors.append(
                f"{prefix}: record_hash "
                "requires source_fields"
            )
            continue

        if (
            mapping.transformation.type
            == "record_hash"
            and target
            != "observation.source_record_id"
        ):
            errors.append(
                f"{prefix}: record_hash is only allowed for "
                "observation.source_record_id"
            )
            continue

        if (
            mapping.transformation.type
            == "sql_expression"
        ):
            expression = (
                mapping.transformation
                .expression
                or ""
            )

            if UNSAFE_EXPRESSION.search(
                expression
            ):
                errors.append(
                    f"{prefix}: unsafe SQL"
                )
                continue

            if target in {
                "entity.abn",
                "entity.acn",
                "entity.nzbn",
            } and re.search(
                r"\b(lpad|rpad)\s*\(",
                expression,
                re.IGNORECASE,
            ):
                errors.append(
                    f"{prefix}: identifier "
                    "padding forbidden"
                )
                continue

        try:
            values = _mapping_values(
                mapping,
                "validation_sample",
            )

        except Exception as e:
            errors.append(
                f"{prefix}: transform failed: {e}"
            )
            continue

        non_null = values[
            ~values.apply(
                is_null
            )
        ]

        invalid = [
            value
            for value in non_null
            if not canonical_value_valid(
                target,
                value,
            )
        ]

        null_count = (
            len(values)
            - len(non_null)
        )

        invalid_count = len(
            invalid
        )

        metrics[target] = {
            "rows": len(values),
            "non_null": len(
                non_null
            ),
            "null_rate": (
                round(
                    null_count
                    / len(values),
                    4,
                )
                if len(values)
                else None
            ),
            "invalid_count": (
                invalid_count
            ),
            "invalid_rate": (
                round(
                    invalid_count
                    / len(non_null),
                    4,
                )
                if len(non_null)
                else 0.0
            ),
            "invalid_examples": [
                str(x)
                for x in invalid[:5]
            ],
        }

        if invalid_count:
            errors.append(
                f"{prefix}: "
                f"{invalid_count} invalid "
                f"values for {target}"
            )

        if (
            target
            in REQUIRED_OBSERVATION_FIELDS
            and null_count
        ):
            errors.append(
                f"{prefix}: required "
                f"{target} has nulls"
            )

        if target == (
            "observation.source_record_id"
        ):
            duplicates = int(
                non_null
                .duplicated()
                .sum()
            )

            metrics[target][
                "duplicate_count"
            ] = duplicates

            if duplicates:
                errors.append(
                    f"{prefix}: "
                    "source_record_id "
                    f"duplicates={duplicates}"
                )

    for target, mappings in targets.items():
        if len(mappings) > 1:
            errors.append(
                "Multiple mappings produce "
                f"{target}"
            )

    mapped_targets = set(
        targets
    )

    runtime_targets = set(
        config.runtime_fields
    )

    for target in (
        REQUIRED_OBSERVATION_FIELDS
    ):
        if (
            target not in mapped_targets
            and target
            not in runtime_targets
        ):
            errors.append(
                "Required observation "
                f"field missing: {target}"
            )

    unmapped = {
        item.source_field
        for item
        in config.unmapped_fields
    }

    unknown_unmapped = (
        unmapped
        - source_columns
    )

    if unknown_unmapped:
        errors.append(
            "Unknown unmapped fields: "
            + str(
                sorted(
                    unknown_unmapped
                )
            )
        )

    unaccounted = (
        source_columns
        - semantic_sources
        - unmapped
    )

    if unaccounted:
        errors.append(
            "Unaccounted source fields: "
            + str(
                sorted(
                    unaccounted
                )
            )
        )

    return json_safe({
        "valid": not errors,
        "errors": errors,
        "warnings": warnings,
        "field_metrics": metrics,
        "validation_rows": (
            CTX.get(
                "validation_rows"
            )
        ),
    })


def _agent_output_from_parts(
    source_reliability: float,
    source_reliability_reason: str,
    mappings: list[FieldMapping],
    unmapped_fields: list[UnmappedField],
    research_notes: list[ResearchNote] | None = None,
    agent_notes: list[str] | None = None,
) -> AgentMappingOutput:
    return AgentMappingOutput(
        source_reliability=source_reliability,
        source_reliability_reason=source_reliability_reason,
        mappings=mappings,
        unmapped_fields=unmapped_fields,
        research_notes=research_notes or [],
        agent_notes=agent_notes or [],
    )


@tool
def submit_candidate(
    source_reliability: float,
    source_reliability_reason: str,
    mappings: list[FieldMapping],
    unmapped_fields: list[UnmappedField],
    research_notes: list[ResearchNote] | None = None,
    agent_notes: list[str] | None = None,
) -> str:
    """
    Validate and submit the Firmable mapping candidate.

    Call this tool when a candidate is ready.
    If accepted=false, fix the returned validation errors and call again.
    A successful agent run should finish only after accepted=true.
    """

    try:
        agent_output = _agent_output_from_parts(
            source_reliability=source_reliability,
            source_reliability_reason=source_reliability_reason,
            mappings=mappings,
            unmapped_fields=unmapped_fields,
            research_notes=research_notes,
            agent_notes=agent_notes,
        )

        config = add_system_envelope(
            agent_output
        )

        validation = validate_config(
            config
        )

        CTX["last_agent_output"] = agent_output
        CTX["last_candidate"] = config
        CTX["last_validation"] = validation

        if validation["valid"]:
            CTX["submitted_candidate"] = config

        return json.dumps(
            json_safe({
                "accepted": bool(
                    validation["valid"]
                ),
                "validation": validation,
            }),
            indent=2,
            allow_nan=False,
        )

    except Exception as e:
        validation = {
            "valid": False,
            "errors": [
                f"{type(e).__name__}: {e}"
            ],
            "warnings": [],
        }

        CTX["last_validation"] = validation

        return json.dumps(
            json_safe({
                "accepted": False,
                "validation": validation,
            }), allow_nan=False,
            indent=2,
        )

## Extractor


In [13]:
def _runtime_observed_at(
    source: SourceDescriptor,
) -> str:

    for value in [
        source.resource_last_modified,
        source.metadata_modified,
    ]:
        if value:
            parsed = pd.to_datetime(
                value,
                errors="coerce",
                utc=True,
            )

            if not pd.isna(
                parsed
            ):
                return parsed.isoformat()

    return pd.Timestamp.now(tz="UTC").isoformat()


def _mapping_expression(
    mapping: FieldMapping,
) -> str:

    transform = (
        mapping.transformation
    )

    if transform.type == "direct":
        return quote_identifier(
            mapping.source_fields[0]
        )

    if transform.type == "sql_expression":
        expression = (
            transform.expression
            or ""
        )

        if UNSAFE_EXPRESSION.search(
            expression
        ):
            raise ValueError(
                "Unsafe SQL expression"
            )

        return expression

    if transform.type == "constant":
        return sql_literal(
            transform.value
        )

    if transform.type == "record_hash":
        pieces = [
            (
                "COALESCE("
                f"CAST({quote_identifier(field)} "
                "AS VARCHAR), '∅')"
            )
            for field
            in mapping.source_fields
        ]

        return (
            "sha256(concat_ws('||', "
            + ", ".join(pieces)
            + "))"
        )

    raise ValueError(
        "Unsupported transformation: "
        + transform.type
    )


def build_gold_select(
    config: MappingConfig,
    source_relation: str,
) -> str:

    mapping_by_target = {
        item.target_field: item
        for item
        in config.mappings
    }

    expressions = []

    observed_at = (
        _runtime_observed_at(
            config.source
        )
    )

    runtime_values = {
        "observation.observed_at":
            observed_at,
        "observation.extractor_version":
            EXTRACTOR_VERSION,
    }

    for target in CANONICAL_FIELDS:
        alias = target.replace(
            ".",
            "_",
        )

        if target in mapping_by_target:
            expression = (
                _mapping_expression(
                    mapping_by_target[
                        target
                    ]
                )
            )

        elif target == "address.country":
            expression = sql_literal(
                (
                    FIELD_SPECS.get(
                        "address.country",
                        {},
                    )
                    or {}
                ).get(
                    "default",
                    "AU",
                )
            )

        elif target == (
            "observation.ingested_at"
        ):
            expression = (
                "CURRENT_TIMESTAMP"
            )

        elif target in runtime_values:
            expression = sql_literal(
                runtime_values[target]
            )

        else:
            expression = "NULL"

        expressions.append(
            f"{expression} AS "
            f"{quote_identifier(alias)}"
        )

    field_confidence = {
        item.target_field:
            item.confidence
        for item
        in config.mappings
    }

    derivation_level = {
        item.target_field:
            item.derivation_level
        for item
        in config.mappings
    }

    if "address.country" not in field_confidence:
        field_confidence["address.country"] = 1.0
        derivation_level["address.country"] = "L1"

    expressions.append(
        sql_literal(
            config.source_reliability
        )
        + " AS "
        + "confidence_source_reliability"
    )

    expressions.append(
        sql_literal(
            json.dumps(
                json_safe(field_confidence), allow_nan=False,
                ensure_ascii=False,
            )
        )
        + " AS field_confidence"
    )

    expressions.append(
        sql_literal(
            json.dumps(
                json_safe(derivation_level), allow_nan=False,
                ensure_ascii=False,
            )
        )
        + " AS derivation_level"
    )

    return (
        "SELECT\n    "
        + ",\n    ".join(
            expressions
        )
        + "\nFROM "
        + source_relation
    )


def preview_candidate(
    config: MappingConfig,
    limit: int | None = None,
) -> pd.DataFrame:
    # Greedily choose the most complete row, then cover fields still missing.
    # Query the full loaded dataset, not just its first eight/three rows.
    relation = quote_identifier(CTX["source_table"]) if CTX.get("source_table") else "validation_sample"
    base_sql = build_gold_select(config, relation)
    with duckdb_lock:
        columns = [item[0] for item in con.execute(base_sql + "\nLIMIT 0").description]
    missing = list(columns)
    records = []
    while missing and (limit is None or len(records) < limit):
        coverage = " + ".join(
            f"CASE WHEN {quote_identifier(column)} IS NOT NULL "
            f"AND trim(CAST({quote_identifier(column)} AS VARCHAR)) <> '' THEN 1 ELSE 0 END"
            for column in missing
        )
        query = f"""
            WITH mapped AS ({base_sql}),
            scored AS (SELECT *, ({coverage}) AS __preview_coverage FROM mapped)
            SELECT * EXCLUDE (__preview_coverage)
            FROM scored WHERE __preview_coverage > 0
            ORDER BY __preview_coverage DESC, CAST(observation_source_record_id AS VARCHAR)
            LIMIT 1
        """
        with duckdb_lock:
            frame = con.execute(query).df()
        if frame.empty:
            break
        record = json_safe(frame.to_dict("records"))[0]
        records.append(record)
        covered = {column for column in missing if record[column] is not None and str(record[column]).strip()}
        if not covered:
            break
        missing = [column for column in missing if column not in covered]
    return pd.DataFrame(records, columns=columns)


def extract_gold(
    config: MappingConfig,
    limit: int = EXTRACT_ROWS,
) -> dict:

    validation = (
        validate_config(
            config
        )
    )

    if not validation[
        "valid"
    ]:
        raise RuntimeError(
            json.dumps(
                json_safe(validation), allow_nan=False,
                indent=2,
            )
        )

    slug = (
        CTX["dataset"][
            "slug"
        ]
    )

    gold_table = (
        f"gold_{slug}"
    )

    source_table = (
        CTX["source_table"]
    )

    sql = (
        build_gold_select(
            config,
            quote_identifier(
                source_table
            ),
        )
        + f"\nLIMIT {int(limit)}"
    )

    with duckdb_lock:
        con.execute(f"""
            CREATE OR REPLACE TABLE
                {quote_identifier(gold_table)}
            AS
            {sql}
        """)

        count = con.execute(
            f"""
            SELECT COUNT(*)
            FROM {quote_identifier(gold_table)}
            """
        ).fetchone()[0]

        sample = con.execute(
            f"""
            SELECT *
            FROM {quote_identifier(gold_table)}
            LIMIT 10
            """
        ).df()

    # Export each approved result before closing its connection.
    GOLD_COPY_DIR.mkdir(parents=True, exist_ok=True)
    copy_path = GOLD_COPY_DIR / f"{gold_table}.parquet"
    with duckdb_lock:
        con.execute(
            f"COPY {quote_identifier(gold_table)} TO {sql_literal(str(copy_path))} (FORMAT PARQUET)"
        )

    CTX[
        "gold_table"
    ] = gold_table

    return json_safe({
        "gold_table": gold_table,
        "rows": count,
        "sample": (
            json_safe(sample.to_dict("records"))
        ),
    })

## Prompts


In [14]:
RESOURCE_PROMPT = """
You select exactly one CKAN resource for Firmable source onboarding.

Rules:
- choose actual record-level business/entity data
- prefer current/live data over historical/help/documentation
- never choose PDF/help/schema/readme when a usable data file exists
- prefer CSV/TSV/XLSX/XLS/JSON/JSONL/XML/ZIP
- return only ResourceChoice
"""


READER_PROMPT = """
You choose the actual downloaded data file and minimal reader options.

Rules:
- choose the business/entity data file, not documentation
- always populate every ReaderDecision field
- for unused nullable fields use null
- for a normal clean CSV use delimiter=null, header=true, skip=0, sheet_name=null, record_tag=null
- XML requires record_tag
- return only ReaderDecision
"""


MAPPING_PROMPT = f"""
You generate the semantic mapping for Firmable Part 2.

You receive:
- source metadata
- reader config
- deterministic profile/sample
- deterministic identifier checks
- optionally the previous candidate and held-out validation failures

Rules:
- output must match AgentMappingOutput
- include all required fields, including empty arrays for research_notes and agent_notes
- every Transformation must include expression and value
- direct: exactly one source field, expression=null, value=null
- record_hash: expression=null, value=null; host code computes the hash from source_fields
- never put a SQL hash expression in record_hash; use sql_expression for custom SQL instead
- sql_expression: expression must contain the scalar DuckDB expression, value=null
- constant: expression=null, value must contain the constant
- every ResearchNote must include source_url; use null when unavailable
- wrong mapping is worse than no mapping
- every source column must either be semantically mapped or listed in unmapped_fields
- low-confidence semantic fields stay unmapped
- never invent ABN, ACN, NZBN, or other identifiers
- never pad identifiers
- use deterministic identifier-check results when available
- direct means exactly one source field
- sql_expression is a deterministic scalar DuckDB expression only
- record_hash is allowed only for observation.source_record_id when no stable unique source key exists
- observation.source_record_id is required
- do not emit observation.source_id or observation.licence; host code adds them
- do not emit observed_at, ingested_at, extractor_version; host code adds them
- source_reliability is separate from field confidence
- link_confidence does not belong in Part 2
- source-specific transformation belongs in config, not dataset-specific Python
- when revising, fix only validation failures unless evidence shows another problem

FIRMABLE ONTOLOGY

{ONTOLOGY_TEXT}
"""


TURBO_PROMPT = f"""
You are the Firmable Turbo schema-mapping agent.

The source is already selected, downloaded, loaded and profiled.
Do NOT restart discovery, download another resource, or rebuild normal onboarding.

You receive:
- current candidate
- deterministic validation output
- human feedback/edits
- source profile and identifier checks

You may use:
- query_source for deeper read-only DuckDB investigation
- run_python for bounded testing
- search_docs only when authoritative external semantics are genuinely needed
- submit_candidate to validate and submit the corrected mapping

Rules:
- wrong mapping is worse than no mapping
- do not weaken validation to make a mapping pass
- do not invent identifiers or unsupported semantics
- keep source-specific logic in the mapping config
- use exact ontology fields
- do not expose API keys
- call submit_candidate before finishing
- if submit_candidate rejects the mapping, fix it and submit again

FIRMABLE ONTOLOGY

{ONTOLOGY_TEXT}
"""

## Agents


In [15]:
def _assert_openai_structured_schema(
    model_class,
):
    schema = model_class.model_json_schema()

    def walk(node, path="$"):
        if isinstance(node, dict):
            if node.get("type") == "object":
                if node.get(
                    "additionalProperties"
                ) is not False:
                    raise RuntimeError(
                        f"{model_class.__name__}: "
                        f"{path} must set "
                        "additionalProperties=false"
                    )

                properties = (
                    node.get(
                        "properties"
                    )
                    or {}
                )

                required = set(
                    node.get(
                        "required"
                    )
                    or []
                )

                missing = (
                    set(properties)
                    - required
                )

                if missing:
                    raise RuntimeError(
                        f"{model_class.__name__}: "
                        f"{path} has non-required "
                        f"properties: {sorted(missing)}"
                    )

            # Empty schema {} inside anyOf means unconstrained Any.
            for key, value in node.items():
                if (
                    key == "anyOf"
                    and isinstance(
                        value,
                        list,
                    )
                    and any(
                        option == {}
                        for option in value
                    )
                ):
                    raise RuntimeError(
                        f"{model_class.__name__}: "
                        f"{path} contains "
                        "unconstrained Any"
                    )

                walk(
                    value,
                    f"{path}.{key}",
                )

        elif isinstance(node, list):
            for index, item in enumerate(node):
                walk(
                    item,
                    f"{path}[{index}]",
                )

    walk(schema)
    return schema


for _schema_model in [
    ResourceChoice,
    ReaderDecision,
    AgentMappingOutput,
]:
    _assert_openai_structured_schema(
        _schema_model
    )

print(
    "Structured schemas: compatible"
)


normal_model = None
turbo_model = None

resource_selector = None
reader_selector = None
mapper = None
turbo_agent = None

TURBO_TOOLS = [
    query_source,
    run_python,
    submit_candidate,
]

if tavily_client is not None:
    TURBO_TOOLS.append(
        search_docs
    )

if openai_client is not None:
    normal_model = init_chat_model(
        NORMAL_MODEL
    )

    turbo_model = init_chat_model(
        TURBO_MODEL
    )

    resource_selector = (
        normal_model
        .with_structured_output(
            ResourceChoice,
            include_raw=True,
        )
    )

    reader_selector = (
        normal_model
        .with_structured_output(
            ReaderDecision,
            include_raw=True,
        )
    )

    mapper = (
        normal_model
        .with_structured_output(
            AgentMappingOutput,
            include_raw=True,
        )
    )

    turbo_agent = create_deep_agent(
        model=turbo_model,
        tools=TURBO_TOOLS,
        system_prompt=TURBO_PROMPT,
    )

print(
    "Normal:",
    "bounded structured pipeline"
    if normal_model is not None
    else "disabled until a valid OPENAI_API_KEY is provided",
)

print(
    "Turbo tools:",
    [
        item.name
        for item in TURBO_TOOLS
    ],
)

Structured schemas: compatible
Normal: bounded structured pipeline
Turbo tools: ['query_source', 'run_python', 'submit_candidate', 'search_docs']


## Streaming


In [16]:
def _render_agent_message(
    message,
    mode: str,
):
    if isinstance(
        message,
        AIMessage,
    ):
        tool_calls = (
            getattr(
                message,
                "tool_calls",
                None,
            )
            or []
        )

        for call in tool_calls:
            print(
                f"[{mode}] → "
                f"{call.get('name', 'tool')}"
            )

        text = message_text(
            message
        ).strip()

        if text:
            print(
                f"[{mode}] "
                f"{text[:3000]}"
            )

    elif isinstance(
        message,
        ToolMessage,
    ):
        name = (
            getattr(
                message,
                "name",
                None,
            )
            or "tool"
        )

        content = message_text(
            message
        ).strip()

        status = (
            "✗"
            if content.startswith(
                "ERROR"
            )
            else "✓"
        )

        print(
            f"[tool] {name} {status}"
        )

        if content.startswith(
            "ERROR"
        ):
            print(
                content[:1500]
            )


def _usage_from_raw(
    raw,
) -> dict:
    usage = (
        getattr(
            raw,
            "usage_metadata",
            None,
        )
        or {}
    )

    return {
        "model_calls": (
            1 if raw is not None else 0
        ),
        "input_tokens": int(
            usage.get(
                "input_tokens",
                0,
            )
            or 0
        ),
        "output_tokens": int(
            usage.get(
                "output_tokens",
                0,
            )
            or 0
        ),
        "total_tokens": int(
            usage.get(
                "total_tokens",
                0,
            )
            or 0
        ),
    }


def _structured_call(
    runnable,
    system_prompt: str,
    payload: dict,
    label: str,
    attempts: int = 2,
):
    last_error = None
    retry_payload = dict(payload)

    if attempts < 1:
        raise ValueError("attempts must be at least 1")

    for attempt in range(
        attempts
    ):
        print(
            f"[normal] {label}"
            + (
                ""
                if attempt == 0
                else f" retry {attempt}"
            )
        )

        try:
            response = runnable.invoke(
                [
                    SystemMessage(
                        content=system_prompt
                    ),
                    HumanMessage(
                        content=json.dumps(
                            json_safe(retry_payload),
                            indent=2,
                            allow_nan=False,
                        )
                    ),
                ],
                config={
                    "tags": [
                        "firmable",
                        "part2",
                        "normal",
                        label,
                    ],
                    "metadata": {
                        "dataset_id": (
                            CTX.get(
                                "dataset",
                                {},
                            ).get(
                                "dataset_id"
                            )
                        ),
                        "resource_id": (
                            CTX.get(
                                "resource_id"
                            )
                        ),
                        "stage": label,
                    },
                },
            )
        except ValidationError as error:
            # The SDK may validate before include_raw returns a response.
            last_error = error
            retry_payload["structured_output_error"] = str(error)
            retry_payload["structured_output_instruction"] = (
                "Correct the schema validation errors and return the complete output. "
                "For direct and record_hash, expression and value must both be null."
            )
            continue

        parsed = response.get(
            "parsed"
        )

        raw = response.get(
            "raw"
        )

        error = response.get(
            "parsing_error"
        )

        if parsed is not None:
            print(
                f"[normal] {label} ✓"
            )
            return (
                parsed,
                _usage_from_raw(
                    raw
                ),
            )

        last_error = error
        retry_payload["structured_output_error"] = str(error)
        retry_payload["structured_output_instruction"] = (
            "Correct the schema validation errors and return the complete output. "
            "For direct and record_hash, expression and value must both be null."
        )

    raise RuntimeError(
        f"{label} structured output failed: "
        f"{last_error}"
    )


def _collect_turbo_metrics(
    messages,
    elapsed_seconds: float,
) -> dict:
    model_calls = 0
    tool_calls = 0
    validation_calls = 0
    input_tokens = 0
    output_tokens = 0
    total_tokens = 0

    for message in messages:
        if not isinstance(
            message,
            AIMessage,
        ):
            continue

        model_calls += 1

        calls = (
            getattr(
                message,
                "tool_calls",
                None,
            )
            or []
        )

        tool_calls += len(
            calls
        )

        validation_calls += sum(
            call.get("name")
            == "submit_candidate"
            for call in calls
        )

        usage = (
            getattr(
                message,
                "usage_metadata",
                None,
            )
            or {}
        )

        input_tokens += int(
            usage.get(
                "input_tokens",
                0,
            )
            or 0
        )

        output_tokens += int(
            usage.get(
                "output_tokens",
                0,
            )
            or 0
        )

        total_tokens += int(
            usage.get(
                "total_tokens",
                0,
            )
            or 0
        )

    return {
        "mode": "turbo",
        "model_calls": model_calls,
        "tool_calls": tool_calls,
        "validation_calls": validation_calls,
        "step_count": (
            model_calls
            + tool_calls
        ),
        "input_tokens": input_tokens,
        "output_tokens": output_tokens,
        "total_tokens": total_tokens,
        "estimated_cost_usd": None,
        "wall_clock_seconds": round(
            elapsed_seconds,
            2,
        ),
        "langsmith_enabled": (
            langsmith_client
            is not None
        ),
        "langsmith_project": (
            os.getenv(
                "LANGSMITH_PROJECT"
            )
            if langsmith_client
            else None
        ),
    }


def stream_turbo_agent(
    prompt: str,
):
    CTX.pop(
        "submitted_candidate",
        None,
    )

    CTX.pop(
        "last_candidate",
        None,
    )

    CTX.pop(
        "last_validation",
        None,
    )

    start = time.perf_counter()

    last_state = {}
    seen_messages = 0

    try:
        for state in turbo_agent.stream(
            {
                "messages": [
                    {
                        "role": "user",
                        "content": prompt,
                    }
                ]
            },
            config={
                "recursion_limit": 30,
                "tags": [
                    "firmable",
                    "part2",
                    "turbo",
                ],
                "metadata": {
                    "dataset_id": (
                        CTX.get(
                            "dataset",
                            {},
                        ).get(
                            "dataset_id"
                        )
                    ),
                    "resource_id": (
                        CTX.get(
                            "resource_id"
                        )
                    ),
                },
            },
            stream_mode="values",
        ):
            last_state = state

            messages = state.get(
                "messages",
                [],
            )

            for message in messages[
                seen_messages:
            ]:
                _render_agent_message(
                    message,
                    "turbo",
                )

            seen_messages = len(
                messages
            )

    except Exception as e:
        if (
            type(e).__name__
            == "GraphRecursionError"
        ):
            print(
                "[turbo] bounded tool budget "
                "reached; returning the best "
                "candidate produced so far."
            )
        else:
            raise

    elapsed = (
        time.perf_counter()
        - start
    )

    messages = last_state.get(
        "messages",
        [],
    )

    candidate = (
        CTX.get(
            "submitted_candidate"
        )
        or CTX.get(
            "last_candidate"
        )
    )

    validation = CTX.get(
        "last_validation"
    )

    metrics = (
        _collect_turbo_metrics(
            messages,
            elapsed,
        )
    )

    return (
        candidate,
        validation,
        metrics,
    )

## Persistence


In [17]:
def save_candidate(
    config: MappingConfig,
):
    save_yaml(
        candidate_path(),
        config.model_dump(
            mode="json"
        ),
    )

    return candidate_path()


def save_state(
    stage: str,
    extra: dict | None = None,
):
    if "dataset" not in CTX:
        return

    payload = {
        "stage": stage,
        "reader_recovery": CTX.get("reader_recovery"),
        "dataset": CTX.get(
            "dataset"
        ),
        "resource_id": CTX.get(
            "resource_id"
        ),
        "file_path": CTX.get(
            "file_path"
        ),
        "source_table": CTX.get(
            "source_table"
        ),
        "profile_path": CTX.get(
            "profile_path"
        ),
        "candidate_path": (
            str(
                candidate_path()
            )
            if "dataset"
            in CTX
            else None
        ),
        "updated_at": (
            pd.Timestamp.now(tz="UTC")
            .isoformat()
        ),
    }

    if extra:
        payload.update(
            extra
        )

    state_path().write_text(
        json.dumps(
            json_safe(payload),
            indent=2,
            allow_nan=False,
        ),
        encoding="utf-8",
    )


def combine_metrics(
    history: list[dict],
) -> dict:

    additive = [
        "model_calls",
        "tool_calls",
        "validation_calls",
        "step_count",
        "input_tokens",
        "output_tokens",
        "total_tokens",
        "wall_clock_seconds",
    ]

    result = {
        "runs": history,
        "revision_count": max(
            0,
            len(history) - 1,
        ),
    }

    for key in additive:
        result[key] = sum(
            float(
                item.get(
                    key,
                    0,
                )
                or 0
            )
            for item in history
        )

    costs = [
        item.get(
            "estimated_cost_usd"
        )
        for item in history
        if item.get(
            "estimated_cost_usd"
        )
        is not None
    ]

    result[
        "estimated_cost_usd"
    ] = (
        round(
            sum(costs),
            6,
        )
        if costs
        else None
    )

    result[
        "langsmith_enabled"
    ] = (
        langsmith_client
        is not None
    )

    result[
        "langsmith_project"
    ] = (
        os.getenv(
            "LANGSMITH_PROJECT"
        )
        if langsmith_client
        else None
    )

    return json_safe(result)

## Agent runner


In [18]:
def _normal_usage_metrics(
    usages: list[dict],
    elapsed_seconds: float,
    validation_attempts: int,
) -> dict:
    model_calls = sum(
        item.get(
            "model_calls",
            0,
        )
        for item in usages
    )

    input_tokens = sum(
        item.get(
            "input_tokens",
            0,
        )
        for item in usages
    )

    output_tokens = sum(
        item.get(
            "output_tokens",
            0,
        )
        for item in usages
    )

    total_tokens = sum(
        item.get(
            "total_tokens",
            0,
        )
        for item in usages
    )

    input_rate = float(
        os.getenv(
            "FIRMABLE_INPUT_COST_PER_M",
            "0",
        )
        or 0
    )

    output_rate = float(
        os.getenv(
            "FIRMABLE_OUTPUT_COST_PER_M",
            "0",
        )
        or 0
    )

    estimated_cost = None

    if input_rate or output_rate:
        estimated_cost = round(
            input_tokens
            / 1_000_000
            * input_rate
            + output_tokens
            / 1_000_000
            * output_rate,
            6,
        )

    return {
        "mode": "normal",
        "model_calls": model_calls,
        "tool_calls": 0,
        "validation_calls": (
            validation_attempts
        ),
        "step_count": (
            model_calls
            + validation_attempts
        ),
        "input_tokens": input_tokens,
        "output_tokens": output_tokens,
        "total_tokens": total_tokens,
        "estimated_cost_usd": (
            estimated_cost
        ),
        "wall_clock_seconds": round(
            elapsed_seconds,
            2,
        ),
        "langsmith_enabled": (
            langsmith_client
            is not None
        ),
        "langsmith_project": (
            os.getenv(
                "LANGSMITH_PROJECT"
            )
            if langsmith_client
            else None
        ),
    }


def _normal_onboard(
    dataset_name: str,
):
    if normal_model is None:
        raise RuntimeError(
            "Normal model is unavailable. "
            "Check OPENAI_API_KEY."
        )

    start = time.perf_counter()
    usages = []
    validation_attempts = 0

    print(
        "[normal] resolve dataset"
    )

    found = json.loads(
        find_dataset.invoke({
            "dataset_name":
                dataset_name
        })
    )

    if not found.get(
        "found"
    ):
        raise LookupError(
            "Dataset not found in Part 1 "
            f"shortlist: {dataset_name}"
        )

    print(
        "[normal] CKAN metadata"
    )

    metadata = json.loads(
        get_dataset_metadata.invoke({})
    )

    resources = json.loads(
        list_resources.invoke({})
    )

    (
        choice,
        usage,
    ) = _structured_call(
        resource_selector,
        RESOURCE_PROMPT,
        {
            "dataset": metadata,
            "resources": resources,
        },
        "choose resource",
    )

    usages.append(
        usage
    )

    available_ids = {
        item.get(
            "resource_id"
        )
        for item in resources
    }

    if (
        choice.resource_id
        not in available_ids
    ):
        raise RuntimeError(
            "Model selected a resource_id "
            "that is not in CKAN metadata: "
            + choice.resource_id
        )

    print(
        "[normal] resource:",
        choice.resource_id,
        "-",
        choice.reason,
    )

    downloaded = json.loads(
        download_resource.invoke({
            "resource_id":
                choice.resource_id
        })
    )

    files = downloaded.get(
        "files",
        [],
    )

    if not files:
        raise RuntimeError(
            "Selected resource produced "
            "no downloadable files."
        )

    inspections = []

    for file_path in files[:12]:
        inspection = (
            inspect_file.invoke({
                "file_path":
                    file_path
            })
        )

        try:
            inspection_obj = (
                json.loads(
                    inspection
                )
            )
        except Exception:
            inspection_obj = {
                "file_path":
                    file_path,
                "inspection":
                    str(
                        inspection
                    )[:3000],
            }

        inspections.append(
            inspection_obj
        )

    reader_payload = {
        "dataset": metadata,
        "resource_choice": (
            choice.model_dump()
        ),
        "files": inspections,
    }

    (
        reader,
        usage,
    ) = _structured_call(
        reader_selector,
        READER_PROMPT,
        reader_payload,
        "choose reader",
    )

    usages.append(
        usage
    )

    allowed_files = {
        str(
            Path(file_path)
        )
        for file_path in files
    }

    if (
        str(
            Path(
                reader.file_path
            )
        )
        not in allowed_files
    ):
        raise RuntimeError(
            "Reader selected a file "
            "outside the downloaded resource: "
            + reader.file_path
        )

    load_error = None

    for reader_attempt in range(
        MAX_READER_REVISIONS + 1
    ):
        print(
            "[normal] load source"
            + (
                ""
                if reader_attempt == 0
                else " retry"
            )
        )

        load_result = (
            load_source.invoke({
                "file_path":
                    reader.file_path,
                "options_json":
                    json.dumps(
                        json_safe(reader.options), allow_nan=False
                    ),
            })
        )

        if not str(
            load_result
        ).startswith(
            "ERROR:"
        ):
            load_error = None
            break

        load_error = str(
            load_result
        )

        if (
            reader_attempt
            >= MAX_READER_REVISIONS
        ):
            break

        (
            reader,
            usage,
        ) = _structured_call(
            reader_selector,
            READER_PROMPT,
            {
                **reader_payload,
                "previous_reader": {
                    **reader.model_dump(),
                    "options": reader.options,
                },
                "load_error":
                    load_error,
                "instruction":
                    (
                        "Fix only the reader "
                        "configuration."
                    ),
            },
            "fix reader",
        )

        usages.append(
            usage
        )

    if load_error:
        raise RuntimeError(
            load_error
        )

    print(
        "[normal] profile source"
    )

    profile = json.loads(
        profile_source.invoke({})
    )

    identifier_checks = (
        deterministic_identifier_checks()
    )

    CTX[
        "identifier_checks"
    ] = identifier_checks

    mapping_payload = {
        "source": (
            source_descriptor()
            .model_dump(
                mode="json"
            )
        ),
        "reader": (
            CTX["reader"]
            .model_dump(
                mode="json"
            )
        ),
        "profile": profile,
        "identifier_checks": (
            identifier_checks
        ),
    }

    candidate = None
    validation = None

    for attempt in range(
        MAX_NORMAL_REVISIONS + 1
    ):
        if attempt:
            mapping_payload[
                "previous_candidate"
            ] = (
                candidate
                .model_dump(
                    mode="json"
                )
            )

            mapping_payload[
                "previous_validation"
            ] = validation

            mapping_payload[
                "instruction"
            ] = (
                "Revise only what is "
                "necessary to fix the "
                "deterministic validation "
                "errors."
            )

        (
            agent_output,
            usage,
        ) = _structured_call(
            mapper,
            MAPPING_PROMPT,
            mapping_payload,
            (
                "map schema"
                if attempt == 0
                else f"revise mapping {attempt}"
            ),
        )

        usages.append(
            usage
        )

        candidate = (
            add_system_envelope(
                agent_output
            )
        )

        validation = (
            validate_config(
                candidate
            )
        )

        validation_attempts += 1

        print(
            "[normal] validation:",
            (
                "PASS"
                if validation["valid"]
                else (
                    "FAIL "
                    f"({len(validation['errors'])} errors)"
                )
            ),
        )

        if validation[
            "valid"
        ]:
            break

    # Bounded by design. Even if Luna cannot self-correct every issue,
    # stop here and let the human choose edit or Turbo.
    elapsed = (
        time.perf_counter()
        - start
    )

    metrics = (
        _normal_usage_metrics(
            usages,
            elapsed,
            validation_attempts,
        )
    )

    CTX[
        "metric_history"
    ] = [
        metrics
    ]

    CTX[
        "last_candidate"
    ] = candidate

    CTX[
        "last_validation"
    ] = validation

    save_candidate(
        candidate
    )

    save_state(
        "review",
        {
            "validation":
                validation,
            "metrics":
                metrics,
        },
    )

    return (
        candidate,
        validation,
        metrics,
    )


def _revise_candidate(
    candidate: MappingConfig,
    validation: dict,
    feedback: str,
    turbo: bool,
):
    if not turbo:
        # Normal edit is intentionally bounded and structured.
        payload = {
            "source": (
                candidate.source
                .model_dump(
                    mode="json"
                )
            ),
            "reader": (
                candidate.reader
                .model_dump(
                    mode="json"
                )
            ),
            "profile": (
                CTX.get(
                    "profile",
                    {}
                )
            ),
            "identifier_checks": (
                CTX.get(
                    "identifier_checks",
                    {}
                )
            ),
            "previous_candidate": (
                candidate
                .model_dump(
                    mode="json"
                )
            ),
            "previous_validation": (
                validation
            ),
            "human_feedback": (
                feedback
            ),
        }

        (
            output,
            usage,
        ) = _structured_call(
            mapper,
            MAPPING_PROMPT,
            payload,
            "human-requested edit",
        )

        revised = (
            add_system_envelope(
                output
            )
        )

        revised_validation = (
            validate_config(
                revised
            )
        )

        metrics = (
            _normal_usage_metrics(
                [usage],
                0.0,
                1,
            )
        )

    else:
        profile = CTX.get(
            "profile",
            {}
        )

        identifier_checks = (
            CTX.get(
                "identifier_checks",
                {}
            )
        )

        prompt = f"""
Fix the current Firmable mapping.

HUMAN FEEDBACK
{feedback or "(no additional text)"}

CURRENT CANDIDATE
{json.dumps(json_safe(candidate.model_dump(mode="json")), allow_nan=False, indent=2)}

CURRENT VALIDATION
{json.dumps(json_safe(validation), allow_nan=False, indent=2)}

SOURCE PROFILE
{json.dumps(json_safe(profile), indent=2, allow_nan=False)}

DETERMINISTIC IDENTIFIER CHECKS
{json.dumps(json_safe(identifier_checks), indent=2, allow_nan=False)}

Investigate only what is necessary.
Use submit_candidate when ready.
"""

        (
            revised,
            revised_validation,
            metrics,
        ) = stream_turbo_agent(
            prompt
        )

        if revised is None:
            revised = candidate

        if revised_validation is None:
            revised_validation = (
                validate_config(
                    revised
                )
            )

    history = list(
        CTX.get(
            "metric_history",
            [],
        )
    )

    history.append(
        metrics
    )

    CTX[
        "metric_history"
    ] = history

    combined = combine_metrics(
        history
    )

    save_candidate(
        revised
    )

    save_state(
        "review",
        {
            "validation":
                revised_validation,
            "metrics":
                combined,
        },
    )

    return (
        revised,
        revised_validation,
        combined,
    )

## Review


In [19]:
def make_review(
    candidate: MappingConfig,
    validation: dict,
    metrics: dict,
) -> dict:

    mappings = []

    mapped_source_fields = []

    for item in candidate.mappings:
        for field in (
            item.source_fields
        ):
            if (
                field
                not in mapped_source_fields
            ):
                mapped_source_fields.append(
                    field
                )

        transform = (
            item.transformation
        )

        mappings.append({
            "source": (
                ", ".join(
                    item.source_fields
                )
                or "(constant/runtime)"
            ),
            "target": (
                item.target_field
            ),
            "transform": (
                transform.type
            ),
            "expression/value": (
                transform.expression
                if transform.expression
                is not None
                else transform.value
            ),
            "confidence": (
                item.confidence
            ),
            "level": (
                item.derivation_level
            ),
            "reason": (
                item.reason
            ),
        })

    source_records = []

    visible_source_fields = [
        field
        for field
        in mapped_source_fields
        if field
        in set(
            CTX.get(
                "source_columns",
                [],
            )
        )
    ]

    if visible_source_fields:
        try:
            select_fields = (
                ", ".join(
                    quote_identifier(
                        field
                    )
                    for field
                    in visible_source_fields
                )
            )

            with duckdb_lock:
                source_df = con.execute(
                    f"""
                    SELECT {select_fields}
                    FROM validation_sample
                    LIMIT 8
                    """
                ).df()

            source_records = (
                json_safe(source_df.to_dict("records"))
            )

        except Exception:
            source_records = []

    preview_error = None
    preview_records = []

    try:
        preview_df = preview_candidate(
            candidate,
            limit=None,
        )

        preview_records = (
            json_safe(preview_df.to_dict("records"))
        )

        record_mapping = next((
            item for item in candidate.mappings if item.target_field == "observation.source_record_id"
        ), None)
        selected_ids = [str(item["observation_source_record_id"]) for item in preview_records
                        if item.get("observation_source_record_id") is not None]
        if visible_source_fields and record_mapping is not None and selected_ids:
            placeholders = ", ".join("?" for _ in selected_ids)
            relation = quote_identifier(CTX["source_table"]) if CTX.get("source_table") else "validation_sample"
            with duckdb_lock:
                source_df = con.execute(
                    f"SELECT {select_fields} FROM {relation} "
                    f"WHERE CAST({_mapping_expression(record_mapping)} AS VARCHAR) IN ({placeholders})",
                    selected_ids,
                ).df()
            source_records = json_safe(source_df.to_dict("records"))

    except Exception as e:
        preview_error = (
            f"{type(e).__name__}: {e}"
        )

    return json_safe({
        "dataset": (
            candidate
            .source
            .dataset_title
        ),
        "resource": {
            "name": (
                candidate
                .source
                .resource_name
            ),
            "resource_id": (
                candidate
                .source
                .resource_id
            ),
            "format": (
                candidate
                .source
                .resource_format
            ),
            "reader": (
                candidate
                .reader
                .model_dump()
            ),
        },
        "source_reliability": (
            candidate.source_reliability
        ),
        "source_reliability_reason": (
            candidate
            .source_reliability_reason
        ),
        "validation": validation,
        "metrics": metrics,
        "mappings": mappings,
        "source_sample": (
            source_records
        ),
        "sample_results": preview_records,
        "gold_columns": list(preview_df.columns) if preview_records else [],
        "gold_columns_without_sample": [
            column for column in preview_df.columns
            if not any(item.get(column) is not None and str(item[column]).strip() for item in preview_records)
        ] if preview_records else [],
        "preview_error": (
            preview_error
        ),
        "unmapped_fields": [
            item.model_dump()
            for item
            in candidate.unmapped_fields
        ],
        "research_notes": [
            item.model_dump()
            for item
            in candidate.research_notes
        ],
        "candidate_file": str(candidate_path()),
        "options": [
            "approve",
            "reject",
            "edit",
            (
                "edit and hand over "
                "to powerful model"
            ),
        ],
    })


def display_review(review: dict):
    validation = review.get("validation") or {}
    mappings = review.get("mappings") or []
    resource = review.get("resource") or {}
    print(f"\nPART 2 — Review mapping for {review.get('dataset', 'unknown dataset')}")
    print("Resource:", resource.get("name"), "| Format:", resource.get("format"))
    print(f"Mapping: {len(mappings)} target fields; {len(review.get('unmapped_fields') or [])} unmapped source fields")
    print("Validation:", "PASS" if validation.get("valid") else "FAIL")
    print(f"Checks: {len(validation.get('errors') or [])} errors; {len(validation.get('warnings') or [])} warnings")
    for error in validation.get("errors") or []:
        print("  Error:", error)
    for warning in (validation.get("warnings") or [])[:3]:
        print("  Warning:", warning)
    print("Validation checks structure and sample values; the question below asks you to confirm field meaning.")
    print("Source reliability:", review.get("source_reliability"))
    print("Reason:", review.get("source_reliability_reason"))

    print("\nProposed field mappings — source → target:")
    for mapping in mappings:
        print(f"  {mapping['source'][:90]} → {mapping['target']} ({mapping['transform']}, confidence {mapping['confidence']:.2f})")
    if review.get("source_sample"):
        source_preview = pd.DataFrame(review["source_sample"])
        print(f"\nOriginal source values — {len(source_preview)} records supporting the gold preview:")
        with pd.option_context("display.max_columns", None, "display.max_colwidth", None):
            display(source_preview.astype(object).where(pd.notna(source_preview), None))
    if review.get("preview_error"):
        print("Mapped preview error:", review["preview_error"])
    elif review.get("sample_results"):
        preview = pd.DataFrame(review["sample_results"], columns=review.get("gold_columns") or None)
        missing = review.get("gold_columns_without_sample") or []
        print(f"\nProposed gold values — {len(preview)} records covering available values across all {len(preview.columns)} schema columns:")
        with pd.option_context("display.max_columns", None, "display.max_colwidth", None, "display.max_rows", None):
            display(preview.astype(object).where(pd.notna(preview), None))
        print(f"Columns with a sample value: {len(preview.columns) - len(missing)}/{len(preview.columns)}")
        if missing:
            print("No non-empty value found in the loaded dataset for: " + ", ".join(missing))
    print("\nCandidate file:", review.get("candidate_file"))
    print("Question: does this mapping correctly represent the source data and produce the gold values you expect?")


## Workflow


In [20]:
class WorkflowState(
    TypedDict,
    total=False,
):
    dataset_name: str
    candidate: dict
    validation: dict
    metrics: dict
    decision: dict
    review: dict
    final: dict
    database_ready: bool
    retry_approved: bool
    review_error: str



def check_approved_node(state: WorkflowState):
    # Resolve locally before opening DuckDB or making any model/network calls.
    found = json.loads(find_dataset.invoke({"dataset_name": state["dataset_name"]}))
    if not found.get("found"):
        raise LookupError(f"Dataset title/ID is missing or ambiguous: {state['dataset_name']}")

    path = approved_path()
    if not path.exists():
        return json_safe({"retry_approved": True})

    decision = interrupt(json_safe({
        "kind": "approved_dataset",
        "dataset": found["title"],
        "mapping_path": str(path),
        "message": "This dataset already has an approved mapping. Retry onboarding?",
    }))
    retry = isinstance(decision, dict) and decision.get("action") == "retry"
    return json_safe({
        "retry_approved": retry,
        "final": {} if retry else {
            "status": "skipped",
            "mapping_path": str(path),
        },
    })


def route_approved(state: WorkflowState):
    return "onboard" if state.get("retry_approved") else END

def onboard_node(
    state: WorkflowState,
):
    if normal_model is None:
        raise RuntimeError("Normal model is unavailable. Check OPENAI_API_KEY.")
    if get_db() is None:
        return {"database_ready": False, "final": {"status": "database_unavailable"}}

    (
        candidate,
        validation,
        metrics,
    ) = _normal_onboard(
        state[
            "dataset_name"
        ]
    )

    return json_safe({
        "database_ready": True,
        "candidate": (
            candidate
            .model_dump(
                mode="json"
            )
        ),
        "validation": (
            validation
        ),
        "metrics": metrics,
    })


def review_node(
    state: WorkflowState,
):
    candidate = (
        MappingConfig
        .model_validate(
            state[
                "candidate"
            ]
        )
    )

    review = make_review(
        candidate,
        state[
            "validation"
        ],
        state[
            "metrics"
        ],
    )

    save_state(
        "human_review",
        {
            "review": review
        },
    )

    review["kind"] = "mapping_review"
    if state.get("review_error"):
        review["edit_error"] = state["review_error"]

    decision = interrupt(
        json_safe(review)
    )

    if isinstance(
        decision,
        str,
    ):
        decision = {
            "action": decision
        }

    return json_safe({
        "decision": decision,
        "review": review,
    })


def route_review(
    state: WorkflowState,
):
    action = (
        (
            state.get(
                "decision"
            )
            or {}
        )
        .get(
            "action",
            "",
        )
        .strip()
        .lower()
    )

    if action == "approve":
        if not (
            state.get(
                "validation"
            )
            or {}
        ).get(
            "valid"
        ):
            return "review"

        return "approve"

    if action == "reject":
        return "reject"

    if action == "edit":
        return "edit"

    if action in {
        "edit_turbo",
        "edit and hand over",
        "edit_and_handover",
        (
            "edit and hand over "
            "to powerful model"
        ),
    }:
        return "turbo"

    return "review"


def _review_candidate_from_decision(
    decision: dict,
    fallback: dict,
) -> MappingConfig:

    mapping = decision.get(
        "mapping"
    )

    if mapping is None:
        path = candidate_path()

        if path.exists():
            mapping = yaml.safe_load(
                path.read_text(
                    encoding="utf-8"
                )
            )
        else:
            mapping = fallback

    if isinstance(
        mapping,
        str,
    ):
        mapping = yaml.safe_load(
            mapping
        )

    return MappingConfig.model_validate(
        mapping
    )


def edit_node(
    state: WorkflowState,
):
    decision = (
        state.get(
            "decision"
        )
        or {}
    )

    try:
        edited = _review_candidate_from_decision(
            decision,
            state["candidate"],
        )
    except (ValidationError, yaml.YAMLError, OSError, ValueError) as error:
        return json_safe({"review_error": str(error)})

    validation = validate_config(
        edited
    )

    save_candidate(
        edited
    )

    save_state(
        "human_review",
        {
            "validation": validation,
            "metrics": (
                state.get(
                    "metrics"
                )
                or {}
            ),
        },
    )

    return json_safe({
        "candidate": (
            edited
            .model_dump(
                mode="json"
            )
        ),
        "validation": validation,
        "review_error": "",
        "metrics": (
            state.get(
                "metrics"
            )
            or {}
        ),
    })


def turbo_node(
    state: WorkflowState,
):
    decision = (
        state.get(
            "decision"
        )
        or {}
    )

    base = _review_candidate_from_decision(
        decision,
        state[
            "candidate"
        ],
    )

    (
        revised,
        validation,
        metrics,
    ) = _revise_candidate(
        base,
        state[
            "validation"
        ],
        decision.get(
            "feedback",
            "",
        ),
        turbo=True,
    )

    return json_safe({
        "candidate": (
            revised
            .model_dump(
                mode="json"
            )
        ),
        "validation": validation,
        "metrics": metrics,
    })
def approve_node(
    state: WorkflowState,
):
    if get_db() is None:
        return {"final": {"status": "database_unavailable"}}

    candidate = (
        MappingConfig
        .model_validate(
            state[
                "candidate"
            ]
        )
    )

    extraction = extract_gold(
        candidate,
        limit=EXTRACT_ROWS,
    )

    save_yaml(
        approved_path(),
        candidate.model_dump(
            mode="json"
        ),
    )


    metrics = (
        state.get(
            "metrics"
        )
        or {}
    )

    metrics_payload = {
        **metrics,
        "dataset": (
            CTX.get(
                "dataset"
            )
        ),
        "resource_id": (
            CTX.get(
                "resource_id"
            )
        ),
        "validation": (
            state.get(
                "validation"
            )
        ),
        "extraction_rows": (
            extraction[
                "rows"
            ]
        ),
        "approved": True,
    }

    metrics_path().write_text(
        json.dumps(
            json_safe(metrics_payload),
            indent=2,
            allow_nan=False,
        ),
        encoding="utf-8",
    )

    save_state(
        "approved",
        {
            "gold_table": (
                extraction[
                    "gold_table"
                ]
            ),
            "rows": (
                extraction[
                    "rows"
                ]
            ),
        },
    )

    return json_safe({
        "final": {
            "status": "approved",
            "mapping_path": str(
                approved_path()
            ),
            "gold_table": (
                extraction[
                    "gold_table"
                ]
            ),
            "rows": (
                extraction[
                    "rows"
                ]
            ),
            "sample": (
                extraction[
                    "sample"
                ]
            ),
        }
    })


def reject_node(
    state: WorkflowState,
):
    candidate = (
        MappingConfig
        .model_validate(
            state[
                "candidate"
            ]
        )
    )

    save_yaml(
        rejected_path(),
        candidate.model_dump(
            mode="json"
        ),
    )

    save_state(
        "rejected"
    )

    return json_safe({
        "final": {
            "status": "rejected",
            "rejected_path": str(
                rejected_path()
            ),
        }
    })


builder = StateGraph(
    WorkflowState
)

builder.add_node(
    "onboard",
    onboard_node,
)

builder.add_node(
    "review",
    review_node,
)

builder.add_node(
    "edit",
    edit_node,
)

builder.add_node(
    "turbo",
    turbo_node,
)

builder.add_node(
    "approve",
    approve_node,
)

builder.add_node(
    "reject",
    reject_node,
)

builder.add_node("check_approved", check_approved_node)
builder.add_edge(START, "check_approved")
builder.add_conditional_edges(
    "check_approved",
    route_approved,
    {"onboard": "onboard", END: END},
)

builder.add_conditional_edges(
    "onboard",
    lambda state: "review" if state.get("database_ready") else END,
    {"review": "review", END: END},
)

builder.add_conditional_edges(
    "review",
    route_review,
    {
        "review": "review",
        "edit": "edit",
        "turbo": "turbo",
        "approve": "approve",
        "reject": "reject",
    },
)

builder.add_edge(
    "edit",
    "review",
)

builder.add_edge(
    "turbo",
    "review",
)

builder.add_edge(
    "approve",
    END,
)

builder.add_edge(
    "reject",
    END,
)

if SqliteSaver is not None:
    checkpoint_connection = (
        sqlite3.connect(
            CHECKPOINT_PATH,
            check_same_thread=False,
        )
    )

    checkpointer = (
        SqliteSaver(
            checkpoint_connection
        )
    )

    print(
        "Checkpoint: SQLite"
    )

else:
    checkpointer = (
        InMemorySaver()
    )

    print(
        "Checkpoint: memory "
        "(install langgraph-checkpoint-sqlite "
        "for persistent resume)"
    )

workflow = builder.compile(
    checkpointer=checkpointer
)

Checkpoint: SQLite


## User API


In [21]:
def _interrupt_payload(
    result,
):
    if not isinstance(
        result,
        dict,
    ):
        return None

    interrupts = result.get(
        "__interrupt__"
    )

    if not interrupts:
        return None

    first = interrupts[0]

    return getattr(
        first,
        "value",
        first,
    )


def _show_workflow_result(
    result,
):
    payload = (
        _interrupt_payload(
            result
        )
    )

    if payload is not None:
        return result

    final = (
        result.get(
            "final"
        )
        if isinstance(
            result,
            dict,
        )
        else None
    )

    if final:
        print("\nFINAL")
        print(
            "Status:",
            final.get(
                "status"
            ),
        )

        if final.get(
            "mapping_path"
        ):
            print(
                "Mapping:",
                final[
                    "mapping_path"
                ],
            )

        if final.get(
            "gold_table"
        ):
            print(
                "Gold table:",
                final[
                    "gold_table"
                ],
            )

            print(
                "Rows:",
                final.get(
                    "rows"
                ),
            )

            print(
                "\nSAMPLE"
            )

            display(
                pd.DataFrame(
                    final.get(
                        "sample"
                    )
                    or []
                )
            )

        if final.get(
            "rejected_path"
        ):
            print(
                "Rejected candidate:",
                final[
                    "rejected_path"
                ],
            )

    return result


def _ask_workflow_decision(payload: dict):
    if payload.get("kind") == "approved_dataset":
        print(f"\nPART 2 — Confirm retry for {payload['dataset']}")
        print("This dataset already has an approved mapping.")
        print("Mapping:", payload["mapping_path"])
        print("Retry runs profiling and AI mapping again, then asks you to review a new candidate.")
        print("Skip keeps the current approved mapping and gold output.")
        print(f"Question: should {payload['dataset']} be processed again?")
        print("1 - Retry onboarding\n2 - Keep existing mapping and skip")
        while True:
            answer = input("Retry this dataset? [1/2, Enter = skip]: ").strip().lower()
            if answer in {"1", "y", "yes", "retry"}:
                return {"action": "retry"}
            if answer in {"", "2", "n", "no", "skip"}:
                return {"action": "skip"}
            print("Please select 1 or 2.")

    display_review(payload)
    if payload.get("edit_error"):
        print("\nThe edited YAML could not be validated:", payload["edit_error"])
    print("1 - Approve: save this mapping and extract its gold table")
    print("2 - Reject: save the rejected candidate and stop this dataset")
    print("3 - Edit Manually: edit the candidate YAML, then validate and review again")
    print("4 - Improve with Turbo Agent: investigate and revise, then review again")
    actions = {"1": "approve", "2": "reject", "3": "edit", "4": "edit_turbo"}
    while True:
        choice = input(f"Review {payload.get('dataset', 'this dataset')} [1-4]: ").strip()
        action = actions.get(choice)
        if action is None:
            print("Please select 1, 2, 3, or 4.")
            continue
        if action == "approve" and (payload.get("edit_error") or not payload.get("validation", {}).get("valid")):
            print("Validation failed. Choose Edit Manually, Improve with Turbo Agent, or Reject.")
            continue
        decision = {"action": action}
        if action == "edit":
            print("Edit and save this candidate YAML:", payload["candidate_file"])
            input("Save your source-to-target mapping changes, then press Enter to validate and review them: ")
        elif action == "edit_turbo":
            decision["feedback"] = input(
                f"For {payload.get('dataset')}, which field or validation issue should Turbo investigate? (Enter = use current checks): "
            ).strip()
        return decision


def _run_workflow(input_value, config):
    # Keep LangGraph's checkpointed interrupts; collect and resume them in this run.
    while True:
        if isinstance(input_value, Command) and isinstance(input_value.resume, dict):
            action = input_value.resume.get("action")
            if action not in {"retry", "skip"} and get_db() is None:
                return {"final": {"status": "database_unavailable"}}
        if isinstance(input_value, dict):
            input_value = json_safe(input_value)
        try:
            result = workflow.invoke(input_value, config=config)
        except Exception as error:
            close_db()
            print(f"PART 2 — Run stopped safely: {type(error).__name__}: {error}")
            return {"final": {"status": "failed", "error": str(error)}}
        payload = _interrupt_payload(result)
        if payload is None:
            close_db()
            return result
        # All review information is already in the checkpointed payload.
        # Release the writer while the user reads/answers; get_db restores views on resume.
        close_db()
        try:
            decision = _ask_workflow_decision(payload)
        except (EOFError, KeyboardInterrupt):
            print("\nWorkflow paused at human input. Run continue_dataset() to continue.")
            return result
        input_value = Command(resume=json_safe(decision))


def continue_dataset():
    """Continue the current checkpoint after a notebook input was interrupted."""
    if ACTIVE_THREAD_ID is None:
        raise RuntimeError("No active workflow. Run run_dataset(...) first.")
    config = {"configurable": {"thread_id": ACTIVE_THREAD_ID}}
    snapshot = workflow.get_state(config)
    for task in snapshot.tasks:
        for pending in task.interrupts:
            decision = _ask_workflow_decision(pending.value)
            return _show_workflow_result(_run_workflow(Command(resume=json_safe(decision)), config))
    return _show_workflow_result(_run_workflow(None, config))


def run_dataset(
    dataset_name: str,
):
    global ACTIVE_THREAD_ID

    reset_context()

    ACTIVE_THREAD_ID = (
        dataset_slug(
            dataset_name
        )
        + "-"
        + uuid.uuid4().hex[:8]
    )

    config = {
        "configurable": {
            "thread_id":
                ACTIVE_THREAD_ID
        }
    }

    try:
        result = _run_workflow(
            {
                "dataset_name":
                    dataset_name
            },
            config,
        )

        return _show_workflow_result(
            result
        )

    except Exception as e:
        if "dataset" in CTX:
            save_state(
                "failed",
                {
                    "error_type":
                        type(e).__name__,
                    "error":
                        str(e),
                },
            )

        close_db()
        print(f"PART 2 — Run stopped safely: {type(e).__name__}: {e}")
        return {"final": {"status": "failed", "error": str(e)}}


def resume_dataset(
    action: str,
    feedback: str = "",
    mapping: dict | str | None = None,
):
    if get_db() is None:
        return {"final": {"status": "database_unavailable"}}

    if ACTIVE_THREAD_ID is None:
        raise RuntimeError(
            "No active dataset review. "
            "Run run_dataset(...) first."
        )

    normalized = (
        action
        .strip()
        .lower()
        .replace(
            "-",
            "_",
        )
    )

    aliases = {
        "edit and hand over to powerful model":
            "edit_turbo",
        "edit and hand over":
            "edit_turbo",
        "edit_and_handover":
            "edit_turbo",
        "turbo":
            "edit_turbo",
    }

    normalized = aliases.get(
        normalized,
        normalized,
    )

    if normalized not in {
        "approve",
        "reject",
        "edit",
        "edit_turbo",
    }:
        raise ValueError(
            "action must be: "
            "approve, reject, edit, edit_turbo"
        )

    decision = {
        "action": normalized,
        "feedback": feedback,
    }

    if mapping is not None:
        decision[
            "mapping"
        ] = mapping

    config = {
        "configurable": {
            "thread_id":
                ACTIVE_THREAD_ID
        }
    }

    result = _run_workflow(
        Command(
            resume=json_safe(decision)
        ),
        config,
    )

    return _show_workflow_result(
        result
    )

def run_datasets(datasets):
    """Run the selected datasets in order, including confirmation and review."""
    results = []
    for number, dataset in enumerate(datasets, start=1):
        print(f"\nDataset {number}/{len(datasets)}: {dataset['title']}")
        result = run_dataset(dataset["dataset_id"])
        results.append({"dataset": dataset["title"], "result": result})
        if _interrupt_payload(result) is not None:
            print("Batch stopped at the pending human input.")
            break
    return results


## Six selected datasets

These are the six sources from the earlier profiling runs. IDs distinguish duplicate catalogue titles.


In [22]:
SIX_DATASETS = [{'dataset_id': '7b8656f9-606d-4337-af29-66b89b2eeefb', 'title': 'ASIC - Company Dataset'}, {'dataset_id': '5bd7fcab-e315-42cb-8daf-50b7efc2027e', 'title': 'ABN Bulk Extract'}, {'dataset_id': '1ca29b03-220d-4c19-b34e-c7932cd4508d', 'title': 'Victorian Government Schools ABNs'}, {'dataset_id': 'b050b242-4487-4306-abf5-07ca073e5594', 'title': 'ACNC Registered Charities'}, {'dataset_id': '6f1cc305-a8b2-4748-bd40-c00754ad6504', 'title': 'Victorian liquor licences by location'}, {'dataset_id': '732d6bf3-2670-4226-9235-789bcf670288', 'title': 'Business establishments location and industry classification'}]

# Check the selection against the Part 1 shortlist before starting.
for dataset in SIX_DATASETS:
    rows = shortlist[shortlist["dataset_id"] == dataset["dataset_id"]]
    assert len(rows) == 1 and rows.iloc[0]["title"] == dataset["title"], dataset

display(pd.DataFrame(SIX_DATASETS))


,dataset_id,title
0,7b8656f9-606d-4337-af29-66b89b2eeefb,ASIC - Company Dataset
1,5bd7fcab-e315-42cb-8daf-50b7efc2027e,ABN Bulk Extract
2,1ca29b03-220d-4c19-b34e-c7932cd4508d,Victorian Government Schools ABNs
3,b050b242-4487-4306-abf5-07ca073e5594,ACNC Registered Charities
4,6f1cc305-a8b2-4748-bd40-c00754ad6504,Victorian liquor licences by location
5,732d6bf3-2670-4226-9235-789bcf670288,Business establishments location and industry ...


## Run all six datasets

Run this cell once. Answer the confirmation and review prompts as they appear. An empty retry answer keeps an existing approved mapping.


In [23]:
assignment_results = run_datasets(SIX_DATASETS)


Dataset 1/6: ASIC - Company Dataset
[normal] resolve dataset
[normal] CKAN metadata
[normal] choose resource
[normal] choose resource ✓
[normal] resource: 5c3914e6-413e-4a2c-b890-bf8efe3eabf2 - Current machine-readable CSV containing ASIC company records; it is preferable to the help PDF and the ZIP marked not machine readable.
[normal] choose reader
[normal] choose reader ✓
[normal] load source
PART 2 — Reader recovery: company_202609.csv; utf-8-sig → UTF-8; original file preserved.
[normal] profile source
[normal] map schema


KeyboardInterrupt: 

## Inspect gold datasets

In [ ]:
db = get_db() if "get_db" in globals() else con
if db is None:
    print("PART 2 — Gold inspection paused: database is in use; saved results are preserved.")
else:
    try:

        gold_tables = (
            db.execute("""
                SELECT table_name
                FROM information_schema.tables
                WHERE table_schema = 'main'
                  AND table_name LIKE 'gold_%'
                ORDER BY table_name
            """)
            .df()["table_name"]
            .tolist()
        )

        print(f"Found {len(gold_tables)} gold tables")

        summary = []

        for table in gold_tables:
            safe_table = table.replace('"', '""')

            row_count = db.execute(
                f'SELECT COUNT(*) FROM "{safe_table}"'
            ).fetchone()[0]

            summary.append({
                "table": table,
                "rows": row_count,
            })

        display(pd.DataFrame(summary))

        flag = True
        for table in gold_tables:
            safe_table = table.replace('"', '""')
            if flag:
                # print schema only for the first table to avoid clutter
                print("\nSCHEMA")
                display(
                    db.execute(
                        f'DESCRIBE "{safe_table}"'
                    ).df()
                )
                flag = False

            print("\n" + "=" * 100)
            print(table)
            print("=" * 100)

            display(
                db.execute(
                    f'''
                    SELECT *
                    FROM "{safe_table}"
                    LIMIT 10
                    '''
                ).df()
            )
    except duckdb.Error as error:
        print("PART 2 — Gold inspection stopped safely:", error)
    finally:
        close_db()


Found 7 gold tables


,table,rows
0,gold_abn_bulk_extract,1000
1,gold_acnc_registered_charities,1000
2,gold_asic_company,1000
3,gold_business_establishments_location_and_indu...,1000
4,gold_caf_restaurant_bistro_seats,1000
5,gold_victorian_government_schools_abns,1000
6,gold_victorian_liquor_licences_by_location,1000



SCHEMA


,column_name,column_type,null,key,default,extra
0,entity_legal_name,INTEGER,YES,None,None,None
1,entity_trading_name,INTEGER,YES,None,None,None
2,entity_abn,VARCHAR,YES,None,None,None
3,entity_acn,INTEGER,YES,None,None,None
4,entity_nzbn,INTEGER,YES,None,None,None
5,entity_entity_type,VARCHAR,YES,None,None,None
6,entity_status,VARCHAR,YES,None,None,None
7,entity_website,INTEGER,YES,None,None,None
8,entity_industry_code,INTEGER,YES,None,None,None
9,entity_date_registered,INTEGER,YES,None,None,None



gold_abn_bulk_extract


...

,entity_legal_name,entity_trading_name,entity_abn,entity_acn,entity_nzbn,entity_entity_type,entity_status,entity_website,entity_industry_code,entity_date_registered,...,observation_source_record_id,observation_observed_at,observation_ingested_at,observation_valid_from,observation_valid_to,observation_licence,observation_extractor_version,confidence_source_reliability,field_confidence,derivation_level
0,<NA>,Cafe 22,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,4511,<NA>,...,d543ef453fbbcff4ec390ad2c0c307ee1084ccdcd56bd1...,2025-06-26T16:24:55.068044+00:00,2026-10-03 07:14:13.959169+10:00,<NA>,<NA>,Other (Open),firmable-part2-v1,0.8,"{""observation.source_record_id"": 0.95, ""entity...","{""observation.source_record_id"": ""L1"", ""entity..."
1,<NA>,Pepperoni's,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,4511,<NA>,...,dd994f64e359c145eebff150bd6f4ce14898afe6053220...,2025-06-26T16:24:55.068044+00:00,2026-10-03 07:14:13.959169+10:00,<NA>,<NA>,Other (Open),firmable-part2-v1,0.8,"{""observation.source_record_id"": 0.95, ""entity...","{""observation.source_record_id"": ""L1"", ""entity..."
2,<NA>,McDonald's,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,4511,<NA>,...,2937794ffed780ef22e72045597e2edc6472eb75e8866d...,2025-06-26T16:24:55.068044+00:00,2026-10-03 07:14:13.959169+10:00,<NA>,<NA>,Other (Open),firmable-part2-v1,0.8,"{""observation.source_record_id"": 0.95, ""entity...","{""observation.source_record_id"": ""L1"", ""entity..."
3,<NA>,Aria Cafe,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,4511,<NA>,...,3a15b4d5fd533cbb0f7ee2f929969141a7089df716127c...,2025-06-26T16:24:55.068044+00:00,2026-10-03 07:14:13.959169+10:00,<NA>,<NA>,Other (Open),firmable-part2-v1,0.8,"{""observation.source_record_id"": 0.95, ""entity...","{""observation.source_record_id"": ""L1"", ""entity..."
4,<NA>,Vita Juice Energy For Life,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,4512,<NA>,...,e5dc0aeaa65802a4556beb621d9b1e89eaae5e51ad86d7...,2025-06-26T16:24:55.068044+00:00,2026-10-03 07:14:13.959169+10:00,<NA>,<NA>,Other (Open),firmable-part2-v1,0.8,"{""observation.source_record_id"": 0.95, ""entity...","{""observation.source_record_id"": ""L1"", ""entity..."
5,<NA>,Kimchi Tray,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,4511,<NA>,...,acc500bf290ac05358b8b54266538f932a9a7799acd284...,2025-06-26T16:24:55.068044+00:00,2026-10-03 07:14:13.959169+10:00,<NA>,<NA>,Other (Open),firmable-part2-v1,0.8,"{""observation.source_record_id"": 0.95, ""entity...","{""observation.source_record_id"": ""L1"", ""entity..."
6,<NA>,Hudsons Coffee,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,4511,<NA>,...,657c44680cb76cf8915458c7bac865ddd604d9988d0ab0...,2025-06-26T16:24:55.068044+00:00,2026-10-03 07:14:13.959169+10:00,<NA>,<NA>,Other (Open),firmable-part2-v1,0.8,"{""observation.source_record_id"": 0.95, ""entity...","{""observation.source_record_id"": ""L1"", ""entity..."
7,<NA>,Cafe 34,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,4511,<NA>,...,298937d5274b071734f6d5c2d156c178edca060814dad1...,2025-06-26T16:24:55.068044+00:00,2026-10-03 07:14:13.959169+10:00,<NA>,<NA>,Other (Open),firmable-part2-v1,0.8,"{""observation.source_record_id"": 0.95, ""entity...","{""observation.source_record_id"": ""L1"", ""entity..."
8,<NA>,Maha Bar & Grill,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,4511,<NA>,...,778f6be5b4aded4226fe6d9291fee4c720ecef602d5e18...,2025-06-26T16:24:55.068044+00:00,2026-10-03 07:14:13.959169+10:00,<NA>,<NA>,Other (Open),firmable-part2-v1,0.8,"{""observation.source_record_id"": 0.95, ""entity...","{""observation.source_record_id"": ""L1"", ""entity..."
9,<NA>,Migo's,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,4511,<NA>,...,11631928140c8b80d5f772705b7d62c6485f425728f02f...,2025-06-26T16:24:55.068044+00:00,2026-10-03 07:14:13.959169+10:00,<NA>,<NA>,Other (Open),firmable-part2-v1,0.8,"{""observation.source_record_id"": 0.95, ""entity...","{""observation.source_record_id"": ""L1"", ""entity..."



gold_victorian_government_schools_abns


,entity_legal_name,entity_trading_name,entity_abn,entity_acn,entity_nzbn,entity_entity_type,entity_status,entity_website,entity_industry_code,entity_date_registered,...,observation_source_record_id,observation_observed_at,observation_ingested_at,observation_valid_from,observation_valid_to,observation_licence,observation_extractor_version,confidence_source_reliability,field_confidence,derivation_level
0,<NA>,<NA>,35954747085,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,...,1,2025-06-25T17:38:22.490113+00:00,2026-10-02 13:38:20.218221+10:00,<NA>,<NA>,Creative Commons Attribution 3.0 Australia,firmable-part2-v1,0.95,"{""entity.abn"": 0.99, ""observation.source_recor...","{""entity.abn"": ""L1"", ""observation.source_recor..."
1,<NA>,<NA>,37051817052,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,...,3,2025-06-25T17:38:22.490113+00:00,2026-10-02 13:38:20.218221+10:00,<NA>,<NA>,Creative Commons Attribution 3.0 Australia,firmable-part2-v1,0.95,"{""entity.abn"": 0.99, ""observation.source_recor...","{""entity.abn"": ""L1"", ""observation.source_recor..."
2,<NA>,<NA>,39744336864,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,...,4,2025-06-25T17:38:22.490113+00:00,2026-10-02 13:38:20.218221+10:00,<NA>,<NA>,Creative Commons Attribution 3.0 Australia,firmable-part2-v1,0.95,"{""entity.abn"": 0.99, ""observation.source_recor...","{""entity.abn"": ""L1"", ""observation.source_recor..."
3,<NA>,<NA>,88819505004,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,...,8,2025-06-25T17:38:22.490113+00:00,2026-10-02 13:38:20.218221+10:00,<NA>,<NA>,Creative Commons Attribution 3.0 Australia,firmable-part2-v1,0.95,"{""entity.abn"": 0.99, ""observation.source_recor...","{""entity.abn"": ""L1"", ""observation.source_recor..."
4,<NA>,<NA>,89340494951,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,...,12,2025-06-25T17:38:22.490113+00:00,2026-10-02 13:38:20.218221+10:00,<NA>,<NA>,Creative Commons Attribution 3.0 Australia,firmable-part2-v1,0.95,"{""entity.abn"": 0.99, ""observation.source_recor...","{""entity.abn"": ""L1"", ""observation.source_recor..."
5,<NA>,<NA>,62403505329,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,...,26,2025-06-25T17:38:22.490113+00:00,2026-10-02 13:38:20.218221+10:00,<NA>,<NA>,Creative Commons Attribution 3.0 Australia,firmable-part2-v1,0.95,"{""entity.abn"": 0.99, ""observation.source_recor...","{""entity.abn"": ""L1"", ""observation.source_recor..."
6,<NA>,<NA>,61829351598,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,...,28,2025-06-25T17:38:22.490113+00:00,2026-10-02 13:38:20.218221+10:00,<NA>,<NA>,Creative Commons Attribution 3.0 Australia,firmable-part2-v1,0.95,"{""entity.abn"": 0.99, ""observation.source_recor...","{""entity.abn"": ""L1"", ""observation.source_recor..."
7,<NA>,<NA>,30126215434,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,...,33,2025-06-25T17:38:22.490113+00:00,2026-10-02 13:38:20.218221+10:00,<NA>,<NA>,Creative Commons Attribution 3.0 Australia,firmable-part2-v1,0.95,"{""entity.abn"": 0.99, ""observation.source_recor...","{""entity.abn"": ""L1"", ""observation.source_recor..."
8,<NA>,<NA>,30751949495,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,...,37,2025-06-25T17:38:22.490113+00:00,2026-10-02 13:38:20.218221+10:00,<NA>,<NA>,Creative Commons Attribution 3.0 Australia,firmable-part2-v1,0.95,"{""entity.abn"": 0.99, ""observation.source_recor...","{""entity.abn"": ""L1"", ""observation.source_recor..."
9,<NA>,<NA>,61834093182,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,...,40,2025-06-25T17:38:22.490113+00:00,2026-10-02 13:38:20.218221+10:00,<NA>,<NA>,Creative Commons Attribution 3.0 Australia,firmable-part2-v1,0.95,"{""entity.abn"": 0.99, ""observation.source_recor...","{""entity.abn"": ""L1"", ""observation.source_recor..."



gold_victorian_liquor_licences_by_location


,entity_legal_name,entity_trading_name,entity_abn,entity_acn,entity_nzbn,entity_entity_type,entity_status,entity_website,entity_industry_code,entity_date_registered,...,observation_source_record_id,observation_observed_at,observation_ingested_at,observation_valid_from,observation_valid_to,observation_licence,observation_extractor_version,confidence_source_reliability,field_confidence,derivation_level
0,ARNOLD BRUCE,UNIVERSITY SKI CLUB,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,...,31100099,2026-09-14T17:48:10.178976+00:00,2026-10-02 13:48:09.834286+10:00,<NA>,<NA>,Creative Commons Attribution 3.0 Australia,firmable-part2-v1,0.9,"{""observation.source_record_id"": 0.98, ""entity...","{""observation.source_record_id"": ""L0"", ""entity..."
1,C S I R SKI CLUB,C S I R SKI CLUB,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,...,31100138,2026-09-14T17:48:10.178976+00:00,2026-10-02 13:48:09.834286+10:00,<NA>,<NA>,Creative Commons Attribution 3.0 Australia,firmable-part2-v1,0.9,"{""observation.source_record_id"": 0.98, ""entity...","{""observation.source_record_id"": ""L0"", ""entity..."
2,SKILIB ALPINE CLUB,SKILIB ALPINE CLUB,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,...,31100154,2026-09-14T17:48:10.178976+00:00,2026-10-02 13:48:09.834286+10:00,<NA>,<NA>,Creative Commons Attribution 3.0 Australia,firmable-part2-v1,0.9,"{""observation.source_record_id"": 0.98, ""entity...","{""observation.source_record_id"": ""L0"", ""entity..."
3,THEODOR HERZL SOCIAL CLUB,THEODOR HERZL SOCIAL CLUB,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,...,31100269,2026-09-14T17:48:10.178976+00:00,2026-10-02 13:48:09.834286+10:00,<NA>,<NA>,Creative Commons Attribution 3.0 Australia,firmable-part2-v1,0.9,"{""observation.source_record_id"": 0.98, ""entity...","{""observation.source_record_id"": ""L0"", ""entity..."
4,GREYTHORN PARK TENNIS CLUB INC,GREYTHORN PARK TENNIS CLUB,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,...,31100277,2026-09-14T17:48:10.178976+00:00,2026-10-02 13:48:09.834286+10:00,<NA>,<NA>,Creative Commons Attribution 3.0 Australia,firmable-part2-v1,0.9,"{""observation.source_record_id"": 0.98, ""entity...","{""observation.source_record_id"": ""L0"", ""entity..."
5,OLOS CO-OPERATIVE SOCIETY LTD,SKI CLUB,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,...,31100609,2026-09-14T17:48:10.178976+00:00,2026-10-02 13:48:09.834286+10:00,<NA>,<NA>,Creative Commons Attribution 3.0 Australia,firmable-part2-v1,0.9,"{""observation.source_record_id"": 0.98, ""entity...","{""observation.source_record_id"": ""L0"", ""entity..."
6,MELBOURNE WALKING CLUB INC,MELBOURNE WALKING CLUB INC,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,...,31100722,2026-09-14T17:48:10.178976+00:00,2026-10-02 13:48:09.834286+10:00,<NA>,<NA>,Creative Commons Attribution 3.0 Australia,firmable-part2-v1,0.9,"{""observation.source_record_id"": 0.98, ""entity...","{""observation.source_record_id"": ""L0"", ""entity..."
7,W M NASH,VICTORIA POLICE ALPINE CLUB,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,...,31101037,2026-09-14T17:48:10.178976+00:00,2026-10-02 13:48:09.834286+10:00,<NA>,<NA>,Creative Commons Attribution 3.0 Australia,firmable-part2-v1,0.9,"{""observation.source_record_id"": 0.98, ""entity...","{""observation.source_record_id"": ""L0"", ""entity..."
8,ARNOLD BRUCE,UNIVERSITY SKI CLUB,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,...,31101207,2026-09-14T17:48:10.178976+00:00,2026-10-02 13:48:09.834286+10:00,<NA>,<NA>,Creative Commons Attribution 3.0 Australia,firmable-part2-v1,0.9,"{""observation.source_record_id"": 0.98, ""entity...","{""observation.source_record_id"": ""L0"", ""entity..."
9,HASTINGS YACHT CLUB INC,HASTINGS YACHT CLUB,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,...,31101566,2026-09-14T17:48:10.178976+00:00,2026-10-02 13:48:09.834286+10:00,<NA>,<NA>,Creative Commons Attribution 3.0 Australia,firmable-part2-v1,0.9,"{""observation.source_record_id"": 0.98, ""entity...","{""observation.source_record_id"": ""L0"", ""entity..."


## Rejection Sample

Find a dataset name from "outputs/part1/ranked_shortlist_50.csv" and run the Agent as follow. 

In [ ]:
# A dataset for rejected mapping review and manual inspection.

DS_NAME = "Annual Report CBS"
result = run_dataset(DS_NAME)


[normal] resolve dataset
[normal] CKAN metadata
[normal] choose resource
[normal] choose resource ✓
[normal] resource: 26031353-0d76-4c6b-8b52-71bdd4cfdc5d - Associations Act data is the strongest fit for business/entity records among the available resources, and it is provided as a structured data file.
[normal] choose reader
[normal] choose reader ✓
[normal] load source
PART 2 — Reader recovery: 2017-18-consumer-and-business-services-associations-act_data.csv; cp1252 → UTF-8; original file preserved.
[normal] profile source
[normal] map schema
[normal] map schema ✓
[normal] validation: PASS

PART 2 — Review mapping for Annual Report CBS
Resource: Associations Act data | Format: XLSX
Mapping: 3 target fields; 5 unmapped source fields
Validation: PASS
Checks: 0 errors; 0 warnings
Validation checks structure and sample values; the question below asks you to confirm field meaning.
Source reliability: 0.9
Reason: Published by the South Australian Attorney-General's Department as an annual

,S9—Annual report,column1,column2,column3,column4
0,De-registered and wound-up,76,88,100,#



Proposed gold values — 1 records covering available values across all 26 schema columns:


,entity_legal_name,entity_trading_name,entity_abn,entity_acn,entity_nzbn,entity_entity_type,entity_status,entity_website,entity_industry_code,entity_date_registered,address_full,address_locality,address_state,address_postcode,address_country,observation_source_id,observation_source_record_id,observation_observed_at,observation_ingested_at,observation_valid_from,observation_valid_to,observation_licence,observation_extractor_version,confidence_source_reliability,field_confidence,derivation_level
0,None,None,None,None,None,None,None,None,None,None,None,None,None,None,AU,26031353-0d76-4c6b-8b52-71bdd4cfdc5d,017f8699e7d52916156c8d2dded46de8a0c8f3f00f0aa0bd0002364d6375a3c9,2018-10-17T23:34:16.095242+00:00,2026-10-02T15:05:34.357113+10:00,None,None,Creative Commons Attribution 3.0 Australia,firmable-part2-v1,0.9,"{""observation.source_record_id"": 1.0, ""observation.source_id"": 1.0, ""observation.licence"": 1.0, ""address.country"": 1.0}","{""observation.source_record_id"": ""L1"", ""observation.source_id"": ""L0"", ""observation.licence"": ""L0"", ""address.country"": ""L1""}"


Columns with a sample value: 10/26
No non-empty value found in the loaded dataset for: entity_legal_name, entity_trading_name, entity_abn, entity_acn, entity_nzbn, entity_entity_type, entity_status, entity_website, entity_industry_code, entity_date_registered, address_full, address_locality, address_state, address_postcode, observation_valid_from, observation_valid_to

Candidate file: /Users/ario/Documents/firmable/outputs/part2/review/annual_report_cbs.candidate.yaml
Question: does this mapping correctly represent the source data and produce the gold values you expect?
1 - Approve: save this mapping and extract its gold table
2 - Reject: save the rejected candidate and stop this dataset
3 - Edit Manually: edit the candidate YAML, then validate and review again
4 - Improve with Turbo Agent: investigate and revise, then review again
[turbo] → query_source
[tool] query_source ✓
[turbo] → submit_candidate
[turbo] The loaded data confirms this is an aggregate statistics table, not a regist

,S9—Annual report,column1,column2,column3,column4
0,De-registered and wound-up,76,88,100,#



Proposed gold values — 1 records covering available values across all 26 schema columns:


,entity_legal_name,entity_trading_name,entity_abn,entity_acn,entity_nzbn,entity_entity_type,entity_status,entity_website,entity_industry_code,entity_date_registered,address_full,address_locality,address_state,address_postcode,address_country,observation_source_id,observation_source_record_id,observation_observed_at,observation_ingested_at,observation_valid_from,observation_valid_to,observation_licence,observation_extractor_version,confidence_source_reliability,field_confidence,derivation_level
0,None,None,None,None,None,None,None,None,None,None,None,None,None,None,AU,26031353-0d76-4c6b-8b52-71bdd4cfdc5d,017f8699e7d52916156c8d2dded46de8a0c8f3f00f0aa0bd0002364d6375a3c9,2018-10-17T23:34:16.095242+00:00,2026-10-02T15:07:09.371831+10:00,None,None,Creative Commons Attribution 3.0 Australia,firmable-part2-v1,0.9,"{""observation.source_record_id"": 1.0, ""observation.source_id"": 1.0, ""observation.licence"": 1.0, ""address.country"": 1.0}","{""observation.source_record_id"": ""L1"", ""observation.source_id"": ""L0"", ""observation.licence"": ""L0"", ""address.country"": ""L1""}"


Columns with a sample value: 10/26
No non-empty value found in the loaded dataset for: entity_legal_name, entity_trading_name, entity_abn, entity_acn, entity_nzbn, entity_entity_type, entity_status, entity_website, entity_industry_code, entity_date_registered, address_full, address_locality, address_state, address_postcode, observation_valid_from, observation_valid_to

Candidate file: /Users/ario/Documents/firmable/outputs/part2/review/annual_report_cbs.candidate.yaml
Question: does this mapping correctly represent the source data and produce the gold values you expect?
1 - Approve: save this mapping and extract its gold table
2 - Reject: save the rejected candidate and stop this dataset
3 - Edit Manually: edit the candidate YAML, then validate and review again
4 - Improve with Turbo Agent: investigate and revise, then review again

FINAL
Status: rejected
Rejected candidate: /Users/ario/Documents/firmable/outputs/part2/rejected/annual_report_cbs.candidate.yaml


You can test the pipeline with any dataset available through the CKAN API, or choose one from the ranked shortlist generated in Part 1:

```text
outputs/part1/ranked_shortlist_50.csv

## Test any candidate Dataset from CKAN by their id.  

In [ ]:
# fb09c883-7139-494c-8adb-875dc7f26040,"Café, restaurant, bistro seats",City of Melbourne,"CSV, GEOJSON, JSON, XLS",0.3283,"Matched keywords: trading name, address; semantic similarity: 0.539" 
# result_2 = run_dataset("fb09c883-7139-494c-8adb-875dc7f26040")

[normal] resolve dataset
[normal] CKAN metadata
[normal] choose resource
[normal] choose resource ✓
[normal] resource: cc7750fc-3c57-43c3-a5f5-c2af85db4c74 - The CSV resource contains record-level café, restaurant, and bistro business data, including names, addresses, industry classifications, and seating capacity.
[normal] choose reader
[normal] choose reader ✓
[normal] load source
[normal] profile source
[normal] map schema
[normal] map schema ✓
[normal] validation: PASS

PART 2 — Review mapping for Café, restaurant, bistro seats
Resource: Café, restaurant, bistro seats CSV | Format: CSV
Mapping: 6 target fields; 12 unmapped source fields
Validation: PASS
Checks: 0 errors; 0 warnings
Validation checks structure and sample values; the question below asks you to confirm field meaning.
Source reliability: 0.8
Reason: City of Melbourne is an authoritative local-government publisher for business seating and premises information within its municipality; records may be historical census obs

,property_id,base_property_id,block_id,census_year,building_address,clue_small_area,trading_name,business_address,industry_anzsic4_code,industry_anzsic4_description,seating_type,number_of_seats,longitude,latitude,location
0,109837,109837,361,2023,268-276 Victoria Street NORTH MELBOURNE VIC 3051,North Melbourne,The Galbi,270 Victoria Street NORTH MELBOURNE VIC 3051,4511,Cafes and Restaurants,Seats - Indoor,45,144.957390153,-37.80580455,"-37.80580455, 144.957390153"



Proposed gold values — 1 records covering available values across all 26 schema columns:


,entity_legal_name,entity_trading_name,entity_abn,entity_acn,entity_nzbn,entity_entity_type,entity_status,entity_website,entity_industry_code,entity_date_registered,address_full,address_locality,address_state,address_postcode,address_country,observation_source_id,observation_source_record_id,observation_observed_at,observation_ingested_at,observation_valid_from,observation_valid_to,observation_licence,observation_extractor_version,confidence_source_reliability,field_confidence,derivation_level
0,None,The Galbi,None,None,None,None,None,None,4511,None,270 Victoria Street NORTH MELBOURNE VIC 3051,None,None,None,AU,cc7750fc-3c57-43c3-a5f5-c2af85db4c74,00011f501e67c3c7b073297129d1845f7373d35b91c364e0bbc5baa07b220d37,2025-06-26T16:24:55.068044+00:00,2026-10-03T07:13:38.047037+10:00,None,None,Other (Open),firmable-part2-v1,0.8,"{""observation.source_record_id"": 0.95, ""entity.trading_name"": 0.98, ""address.full"": 0.96, ""entity.industry_code"": 0.98, ""observation.source_id"": 1.0, ""observation.licence"": 1.0, ""address.country"": 1.0}","{""observation.source_record_id"": ""L1"", ""entity.trading_name"": ""L0"", ""address.full"": ""L0"", ""entity.industry_code"": ""L0"", ""observation.source_id"": ""L0"", ""observation.licence"": ""L0"", ""address.country"": ""L1""}"


Columns with a sample value: 13/26
No non-empty value found in the loaded dataset for: entity_legal_name, entity_abn, entity_acn, entity_nzbn, entity_entity_type, entity_status, entity_website, entity_date_registered, address_locality, address_state, address_postcode, observation_valid_from, observation_valid_to

Candidate file: /Users/ario/Documents/firmable/outputs/part2/review/caf_restaurant_bistro_seats.candidate.yaml
Question: does this mapping correctly represent the source data and produce the gold values you expect?
1 - Approve: save this mapping and extract its gold table
2 - Reject: save the rejected candidate and stop this dataset
3 - Edit Manually: edit the candidate YAML, then validate and review again
4 - Improve with Turbo Agent: investigate and revise, then review again

FINAL
Status: approved
Mapping: /Users/ario/Documents/firmable/outputs/part2/mapping_configs/caf_restaurant_bistro_seats.mapping.yaml
Gold table: gold_caf_restaurant_bistro_seats
Rows: 1000

SAMPLE


,entity_legal_name,entity_trading_name,entity_abn,entity_acn,entity_nzbn,entity_entity_type,entity_status,entity_website,entity_industry_code,entity_date_registered,...,observation_source_record_id,observation_observed_at,observation_ingested_at,observation_valid_from,observation_valid_to,observation_licence,observation_extractor_version,confidence_source_reliability,field_confidence,derivation_level
0,None,Cafe 22,None,None,None,None,None,None,4511,None,...,d543ef453fbbcff4ec390ad2c0c307ee1084ccdcd56bd1...,2025-06-26T16:24:55.068044+00:00,2026-10-03T07:14:13.959169+10:00,None,None,Other (Open),firmable-part2-v1,0.8,"{""observation.source_record_id"": 0.95, ""entity...","{""observation.source_record_id"": ""L1"", ""entity..."
1,None,Pepperoni's,None,None,None,None,None,None,4511,None,...,dd994f64e359c145eebff150bd6f4ce14898afe6053220...,2025-06-26T16:24:55.068044+00:00,2026-10-03T07:14:13.959169+10:00,None,None,Other (Open),firmable-part2-v1,0.8,"{""observation.source_record_id"": 0.95, ""entity...","{""observation.source_record_id"": ""L1"", ""entity..."
2,None,McDonald's,None,None,None,None,None,None,4511,None,...,2937794ffed780ef22e72045597e2edc6472eb75e8866d...,2025-06-26T16:24:55.068044+00:00,2026-10-03T07:14:13.959169+10:00,None,None,Other (Open),firmable-part2-v1,0.8,"{""observation.source_record_id"": 0.95, ""entity...","{""observation.source_record_id"": ""L1"", ""entity..."
3,None,Aria Cafe,None,None,None,None,None,None,4511,None,...,3a15b4d5fd533cbb0f7ee2f929969141a7089df716127c...,2025-06-26T16:24:55.068044+00:00,2026-10-03T07:14:13.959169+10:00,None,None,Other (Open),firmable-part2-v1,0.8,"{""observation.source_record_id"": 0.95, ""entity...","{""observation.source_record_id"": ""L1"", ""entity..."
4,None,Vita Juice Energy For Life,None,None,None,None,None,None,4512,None,...,e5dc0aeaa65802a4556beb621d9b1e89eaae5e51ad86d7...,2025-06-26T16:24:55.068044+00:00,2026-10-03T07:14:13.959169+10:00,None,None,Other (Open),firmable-part2-v1,0.8,"{""observation.source_record_id"": 0.95, ""entity...","{""observation.source_record_id"": ""L1"", ""entity..."
5,None,Kimchi Tray,None,None,None,None,None,None,4511,None,...,acc500bf290ac05358b8b54266538f932a9a7799acd284...,2025-06-26T16:24:55.068044+00:00,2026-10-03T07:14:13.959169+10:00,None,None,Other (Open),firmable-part2-v1,0.8,"{""observation.source_record_id"": 0.95, ""entity...","{""observation.source_record_id"": ""L1"", ""entity..."
6,None,Hudsons Coffee,None,None,None,None,None,None,4511,None,...,657c44680cb76cf8915458c7bac865ddd604d9988d0ab0...,2025-06-26T16:24:55.068044+00:00,2026-10-03T07:14:13.959169+10:00,None,None,Other (Open),firmable-part2-v1,0.8,"{""observation.source_record_id"": 0.95, ""entity...","{""observation.source_record_id"": ""L1"", ""entity..."
7,None,Cafe 34,None,None,None,None,None,None,4511,None,...,298937d5274b071734f6d5c2d156c178edca060814dad1...,2025-06-26T16:24:55.068044+00:00,2026-10-03T07:14:13.959169+10:00,None,None,Other (Open),firmable-part2-v1,0.8,"{""observation.source_record_id"": 0.95, ""entity...","{""observation.source_record_id"": ""L1"", ""entity..."
8,None,Maha Bar & Grill,None,None,None,None,None,None,4511,None,...,778f6be5b4aded4226fe6d9291fee4c720ecef602d5e18...,2025-06-26T16:24:55.068044+00:00,2026-10-03T07:14:13.959169+10:00,None,None,Other (Open),firmable-part2-v1,0.8,"{""observation.source_record_id"": 0.95, ""entity...","{""observation.source_record_id"": ""L1"", ""entity..."
9,None,Migo's,None,None,None,None,None,None,4511,None,...,11631928140c8b80d5f772705b7d62c6485f425728f02f...,2025-06-26T16:24:55.068044+00:00,2026-10-03T07:14:13.959169+10:00,None,None,Other (Open),firmable-part2-v1,0.8,"{""observation.source_record_id"": 0.95, ""entity...","{""observation.source_record_id"": ""L1"", ""entity..."
